In [2]:
# Cell 1: Install Required Dependencies and Imports
# This cell sets up all necessary libraries for chunk generation

# Install required packages (if needed)
# !pip install --quiet python-magic

import os
import re
import json
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import hashlib

print("✅ All dependencies imported successfully!")
print("🎯 Sequential Chunk Generation System Ready")

✅ All dependencies imported successfully!
🎯 Sequential Chunk Generation System Ready


# Saving file for usage

In [3]:
def create_enhanced_sample_sv_file(filename: str = "RISCV.sv") -> str:
    """Create enhanced sample with more rare regions for testing"""
    enhanced_code = '''// rv32i_5stage_pipelined_core.v
// A compact but fairly complete RV32I 5‑stage pipelined RISC‑V core in synthesizable Verilog-2001.
// Stages: IF, ID, EX, MEM, WB
// Features:
//  - RV32I base ISA (no CSR/ECALL/EBREAK/FENCE)
//  - 32 x 32-bit register file (x0 hard-wired to 0)
//  - Immediate generator (I/S/B/U/J)
//  - Control unit + ALU
//  - Simple static branch prediction (predict-not-taken)
//  - Forwarding unit (EX/MEM, MEM/WB) and hazard detector (load-use stall)
//  - Byte/half/word loads and stores with sign/zero extension
//  - Simple valid/ready handshake instruction + data memory interfaces
//
// Memory interfaces (synchronous request, ready/valid response):
//  Instruction port (read-only):
//    imem_req   : core requests a fetch for PC
//    imem_addr  : address (aligned)
//    imem_rvalid: instruction return valid
//    imem_rdata : 32-bit instruction word
//  Data port (read/write):
//    dmem_req   : request (assert for the duration of a transfer)
//    dmem_we    : 1=write (store), 0=read (load)
//    dmem_be    : byte enables for write {3:0}
//    dmem_addr  : byte address
//    dmem_wdata : write data
//    dmem_rvalid: read data valid
//    dmem_rdata : read data
//    dmem_ready : target can accept request (1-cycle handshake)
//
// Notes:
//  - This file contains multiple modules for clarity and single-file use.
//  - For brevity, there are no CSRs/interrupts/multiplication/atomics/compressed.
//  - Reset is synchronous active-high.

`timescale 1ns/1ps

// ===================== Top Core =====================
module rv32i_core (
    input              clk,
    input              rst,
    // Instruction memory (read-only)
    output             imem_req,
    output     [31:0]  imem_addr,
    input              imem_rvalid,
    input      [31:0]  imem_rdata,
    // Data memory (read/write)
    output             dmem_req,
    output             dmem_we,
    output     [3:0]   dmem_be,
    output     [31:0]  dmem_addr,
    output     [31:0]  dmem_wdata,
    input              dmem_ready,
    input              dmem_rvalid,
    input      [31:0]  dmem_rdata
);
    // ---------- IF stage ----------
    reg  [31:0] pc_if;
    wire [31:0] pc_next;
    wire        take_branch_ex;
    wire [31:0] branch_target_ex;

    // predict not taken: next PC = PC+4 unless branch/jump resolved in EX
    assign pc_next = take_branch_ex ? branch_target_ex : (pc_if + 32'd4);

    // IF/ID pipeline reg
    reg  [31:0] ifid_pc, ifid_instr;
    reg         ifid_valid;

    // Instruction fetch request when pipeline wants next instruction
    assign imem_req  = 1'b1; // always fetching sequentially (simple interface)
    assign imem_addr = pc_if;

    // ---------- ID stage wires ----------
    wire [4:0]  id_rs1, id_rs2, id_rd;
    wire [31:0] id_rdata1, id_rdata2, id_imm;
    wire        id_use_rs1, id_use_rs2;
    wire [2:0]  id_funct3;
    wire [6:0]  id_opcode;
    wire        id_is_branch, id_is_jal, id_is_jalr, id_memread, id_memwrite;
    wire        id_regwrite;
    wire [3:0]  id_aluop;
    wire        id_alusrc;
    wire [1:0]  id_wb_sel; // 0:ALU, 1:MEM, 2:PC+4

    // ---------- EX stage wires ----------
    reg  [31:0] idex_pc, idex_r1, idex_r2, idex_imm;
    reg  [4:0]  idex_rs1, idex_rs2, idex_rd;
    reg  [2:0]  idex_funct3;
    reg  [3:0]  idex_aluop;
    reg         idex_alusrc;
    reg         idex_is_branch, idex_is_jal, idex_is_jalr;
    reg         idex_memread, idex_memwrite;
    reg         idex_regwrite;
    reg  [1:0]  idex_wb_sel;
    reg         idex_valid;

    wire [31:0] ex_alu_b;
    wire [31:0] ex_alu_y;
    wire        ex_br_taken;
    wire [31:0] ex_pc_plus4 = idex_pc + 32'd4;

    // forwarding
    wire [1:0]  fwd_a_sel, fwd_b_sel;
    wire [31:0] ex_op_a, ex_op_b_raw, ex_op_b;

    // ---------- MEM stage regs ----------
    reg  [31:0] exmem_alu_y, exmem_store_data, exmem_pc_plus4;
    reg  [4:0]  exmem_rd;
    reg  [2:0]  exmem_funct3;
    reg         exmem_memread, exmem_memwrite;
    reg         exmem_regwrite;
    reg  [1:0]  exmem_wb_sel;
    reg         exmem_valid;

    // ---------- WB stage regs ----------
    reg  [31:0] memwb_alu_y, memwb_mem_rdata, memwb_pc_plus4;
    reg  [4:0]  memwb_rd;
    reg         memwb_regwrite;
    reg  [1:0]  memwb_wb_sel;
    reg         memwb_valid;

    // ---------- Register File ----------
    regfile rf(
        .clk(clk),
        .rs1(id_rs1), .rs2(id_rs2), .rd(memwb_rd),
        .rd_we(memwb_regwrite & memwb_valid),
        .rd_wdata(
            (memwb_wb_sel==2'd0) ? memwb_alu_y :
            (memwb_wb_sel==2'd1) ? memwb_mem_rdata :
            (memwb_wb_sel==2'd2) ? memwb_pc_plus4 : 32'b0
        ),
        .rs1_rdata(id_rdata1),
        .rs2_rdata(id_rdata2)
    );

    // ---------- Instruction decode ----------
    assign id_opcode = ifid_instr[6:0];
    assign id_rd     = ifid_instr[11:7];
    assign id_funct3 = ifid_instr[14:12];
    assign id_rs1    = ifid_instr[19:15];
    assign id_rs2    = ifid_instr[24:20];

    immgen immg(.instr(ifid_instr), .imm(id_imm));
    control ctrl(
        .opcode(id_opcode), .funct3(id_funct3), .funct7(ifid_instr[31:25]),
        .use_rs1(id_use_rs1), .use_rs2(id_use_rs2),
        .is_branch(id_is_branch), .is_jal(id_is_jal), .is_jalr(id_is_jalr),
        .memread(id_memread), .memwrite(id_memwrite),
        .regwrite(id_regwrite), .aluop(id_aluop), .alusrc(id_alusrc), .wb_sel(id_wb_sel)
    );

    // ---------- Hazard detection (load-use) ----------
    wire load_use_hazard = idex_memread & idex_valid & ifid_valid & (
        (id_use_rs1 && (idex_rd != 5'd0) && (idex_rd == id_rs1)) ||
        (id_use_rs2 && (idex_rd != 5'd0) && (idex_rd == id_rs2))
    );

    // ---------- Forwarding ----------
    forwarding_unit fwd(
        .idex_rs1(idex_rs1), .idex_rs2(idex_rs2),
        .exmem_rd(exmem_rd), .exmem_regwrite(exmem_regwrite & exmem_valid),
        .memwb_rd(memwb_rd), .memwb_regwrite(memwb_regwrite & memwb_valid),
        .fwd_a(fwd_a_sel), .fwd_b(fwd_b_sel)
    );

    // select forwarded operands
    assign ex_op_a = (fwd_a_sel==2'd0) ? idex_r1 :
                     (fwd_a_sel==2'd1) ? exmem_alu_y :
                     (fwd_a_sel==2'd2) ? ((memwb_wb_sel==2'd0)?memwb_alu_y:(memwb_wb_sel==2'd1?memwb_mem_rdata:memwb_pc_plus4)) :
                     idex_r1;

    assign ex_op_b_raw = (fwd_b_sel==2'd0) ? idex_r2 :
                         (fwd_b_sel==2'd1) ? exmem_alu_y :
                         (fwd_b_sel==2'd2) ? ((memwb_wb_sel==2'd0)?memwb_alu_y:(memwb_wb_sel==2'd1?memwb_mem_rdata:memwb_pc_plus4)) :
                         idex_r2;

    assign ex_alu_b = idex_alusrc ? idex_imm : ex_op_b_raw;

    alu alu0(
        .op(idex_aluop), .a(ex_op_a), .b(ex_alu_b), .y(ex_alu_y)
    );

    // Branch unit
    branch_unit bru(
        .funct3(idex_funct3), .rs1(ex_op_a), .rs2(ex_op_b_raw),
        .taken(ex_br_taken)
    );

    assign take_branch_ex   = (idex_is_branch & ex_br_taken) | idex_is_jal | idex_is_jalr;
    assign branch_target_ex = idex_is_jalr ? (ex_op_a + idex_imm) & ~32'd1 : // JALR target (clear LSB)
                              idex_is_jal  ? (idex_pc + idex_imm) :
                              idex_is_branch ? (idex_pc + idex_imm) : 32'b0;

    // ---------- Data memory interface ----------
    // Form write strobes and data according to size in funct3
    wire [3:0] store_be;
    wire [31:0] store_wdata;
    store_align salign(
        .addr(exmem_alu_y[1:0]), .funct3(exmem_funct3), .rs2(exmem_store_data),
        .be(store_be), .wdata(store_wdata)
    );

    assign dmem_req   = exmem_valid & (exmem_memread | exmem_memwrite);
    assign dmem_we    = exmem_memwrite;
    assign dmem_be    = store_be;
    assign dmem_addr  = exmem_alu_y;
    assign dmem_wdata = store_wdata;

    // Load data alignment / sign extend
    wire [31:0] load_data_aligned;
    load_align lalign(
        .addr(exmem_alu_y[1:0]), .funct3(exmem_funct3), .rdata(dmem_rdata),
        .y(load_data_aligned)
    );

    // ---------- Pipeline control (stalls/flushes) ----------
    wire if_stall  = load_use_hazard; // hold IF when hazard
    wire id_stall  = load_use_hazard; // hold ID when hazard
    wire id_flush  = take_branch_ex;  // flush ID on taken branch/jump
    wire ex_flush  = take_branch_ex;  // flush EX control when redirecting

    // ---------- Sequential logic ----------
    always @(posedge clk) begin
        if (rst) begin
            pc_if      <= 32'h0000_0000;
            ifid_pc    <= 32'b0;
            ifid_instr <= 32'b0;
            ifid_valid <= 1'b0;

            // Clear pipeline regs
            idex_pc        <= 32'b0; idex_r1 <= 32'b0; idex_r2 <= 32'b0; idex_imm <= 32'b0;
            idex_rs1       <= 5'b0;  idex_rs2<= 5'b0;  idex_rd  <= 5'b0; idex_funct3 <= 3'b0;
            idex_aluop     <= 4'b0;  idex_alusrc <= 1'b0;
            idex_is_branch <= 1'b0;  idex_is_jal <= 1'b0; idex_is_jalr <= 1'b0;
            idex_memread   <= 1'b0;  idex_memwrite <= 1'b0; idex_regwrite <= 1'b0; idex_wb_sel <= 2'b0;
            idex_valid     <= 1'b0;

            exmem_alu_y    <= 32'b0; exmem_store_data <= 32'b0; exmem_pc_plus4 <= 32'b0;
            exmem_rd       <= 5'b0; exmem_funct3 <= 3'b0;
            exmem_memread  <= 1'b0; exmem_memwrite <= 1'b0; exmem_regwrite <= 1'b0; exmem_wb_sel <= 2'b0;
            exmem_valid    <= 1'b0;

            memwb_alu_y    <= 32'b0; memwb_mem_rdata <= 32'b0; memwb_pc_plus4 <= 32'b0;
            memwb_rd       <= 5'b0; memwb_regwrite <= 1'b0; memwb_wb_sel <= 2'b0; memwb_valid <= 1'b0;
        end else begin
            // PC update
            if (!if_stall) begin
                pc_if <= pc_next;
            end

            // IF/ID register update
            if (id_flush) begin
                ifid_valid <= 1'b0; // bubble
                ifid_pc    <= 32'b0;
                ifid_instr <= 32'b0;
            end else if (!id_stall) begin
                ifid_valid <= imem_rvalid;
                ifid_pc    <= pc_if;     // PC of fetched instruction
                ifid_instr <= imem_rdata;
            end

            // ID/EX register update
            if (ex_flush) begin
                idex_valid     <= 1'b0;
                idex_pc        <= 32'b0;
                idex_r1        <= 32'b0;
                idex_r2        <= 32'b0;
                idex_imm       <= 32'b0;
                idex_rs1       <= 5'b0; idex_rs2 <= 5'b0; idex_rd <= 5'b0;
                idex_funct3    <= 3'b0; idex_aluop <= 4'b0; idex_alusrc <= 1'b0;
                idex_is_branch <= 1'b0; idex_is_jal <= 1'b0; idex_is_jalr <= 1'b0;
                idex_memread   <= 1'b0; idex_memwrite <= 1'b0; idex_regwrite <= 1'b0; idex_wb_sel <= 2'b0;
            end else begin
                if (load_use_hazard) begin
                    // insert bubble by zeroing control while holding IF/ID (done above)
                    idex_valid     <= 1'b0;
                    idex_pc        <= 32'b0;
                    idex_r1        <= 32'b0;
                    idex_r2        <= 32'b0;
                    idex_imm       <= 32'b0;
                    idex_rs1       <= 5'b0; idex_rs2 <= 5'b0; idex_rd <= 5'b0;
                    idex_funct3    <= 3'b0; idex_aluop <= 4'b0; idex_alusrc <= 1'b0;
                    idex_is_branch <= 1'b0; idex_is_jal <= 1'b0; idex_is_jalr <= 1'b0;
                    idex_memread   <= 1'b0; idex_memwrite <= 1'b0; idex_regwrite <= 1'b0; idex_wb_sel <= 2'b0;
                end else begin
                    idex_valid     <= ifid_valid;
                    idex_pc        <= ifid_pc;
                    idex_r1        <= id_rdata1;
                    idex_r2        <= id_rdata2;
                    idex_imm       <= id_imm;
                    idex_rs1       <= id_rs1; idex_rs2 <= id_rs2; idex_rd <= id_rd;
                    idex_funct3    <= id_funct3;
                    idex_aluop     <= id_aluop;
                    idex_alusrc    <= id_alusrc;
                    idex_is_branch <= id_is_branch; idex_is_jal <= id_is_jal; idex_is_jalr <= id_is_jalr;
                    idex_memread   <= id_memread; idex_memwrite <= id_memwrite;
                    idex_regwrite  <= id_regwrite; idex_wb_sel <= id_wb_sel;
                end
            end

            // EX/MEM register update
            exmem_valid     <= idex_valid;
            exmem_alu_y     <= ex_alu_y;
            exmem_store_data<= ex_op_b_raw;
            exmem_pc_plus4  <= ex_pc_plus4;
            exmem_rd        <= idex_rd;
            exmem_funct3    <= idex_funct3;
            exmem_memread   <= idex_memread;
            exmem_memwrite  <= idex_memwrite;
            exmem_regwrite  <= idex_regwrite;
            exmem_wb_sel    <= idex_wb_sel;

            // MEM/WB register update
            memwb_valid     <= exmem_valid;
            memwb_alu_y     <= exmem_alu_y;
            memwb_pc_plus4  <= exmem_pc_plus4;
            memwb_rd        <= exmem_rd;
            memwb_regwrite  <= exmem_regwrite;
            memwb_wb_sel    <= exmem_wb_sel;

            // capture memory read data when valid
            if (dmem_rvalid) begin
                memwb_mem_rdata <= load_data_aligned;
            end
        end
    end

endmodule

// ===================== Register File =====================
module regfile (
    input              clk,
    input      [4:0]   rs1,
    input      [4:0]   rs2,
    input      [4:0]   rd,
    input              rd_we,
    input      [31:0]  rd_wdata,
    output     [31:0]  rs1_rdata,
    output     [31:0]  rs2_rdata
);
    reg [31:0] regs[31:0];

    // read (combinational)
    assign rs1_rdata = (rs1 == 5'd0) ? 32'b0 : regs[rs1];
    assign rs2_rdata = (rs2 == 5'd0) ? 32'b0 : regs[rs2];

    // write (x0 is hardwired to 0)
    always @(posedge clk) begin
        if (rd_we && (rd != 5'd0)) regs[rd] <= rd_wdata;
        regs[0] <= 32'b0;
    end
endmodule

// ===================== Immediate Generator =====================
module immgen(
    input  [31:0] instr,
    output [31:0] imm
);
    wire [6:0] opcode = instr[6:0];
    wire [31:0] I = {{21{instr[31]}}, instr[30:20]};
    wire [31:0] S = {{21{instr[31]}}, instr[30:25], instr[11:7]};
    wire [31:0] B = {{20{instr[31]}}, instr[7], instr[30:25], instr[11:8], 1'b0};
    wire [31:0] U = {instr[31:12], 12'b0};
    wire [31:0] J = {{12{instr[31]}}, instr[19:12], instr[20], instr[30:21], 1'b0};

    assign imm = (opcode==7'b0010011 /*I-type ALU*/ || opcode==7'b0000011 /*LOAD*/ || opcode==7'b1100111 /*JALR*/)
                 ? I :
                 (opcode==7'b0100011 /*STORE*/) ? S :
                 (opcode==7'b1100011 /*BRANCH*/) ? B :
                 (opcode==7'b0110111 /*LUI*/ || opcode==7'b0010111 /*AUIPC*/) ? U :
                 (opcode==7'b1101111 /*JAL*/) ? J : 32'b0;
endmodule

// ===================== Control Unit =====================
module control(
    input  [6:0] opcode,
    input  [2:0] funct3,
    input  [6:0] funct7,
    output       use_rs1,
    output       use_rs2,
    output       is_branch,
    output       is_jal,
    output       is_jalr,
    output       memread,
    output       memwrite,
    output       regwrite,
    output [3:0] aluop,
    output       alusrc,
    output [1:0] wb_sel
);
    // defaults
    reg r_use_rs1, r_use_rs2, r_branch, r_jal, r_jalr, r_memread, r_memwrite, r_regwrite, r_alusrc;
    reg [3:0] r_aluop;
    reg [1:0] r_wbsel;

    always @(*) begin
        r_use_rs1=0; r_use_rs2=0; r_branch=0; r_jal=0; r_jalr=0; r_memread=0; r_memwrite=0; r_regwrite=0; r_alusrc=0; r_aluop=4'h0; r_wbsel=2'd0;
        case (opcode)
            7'b0110011: begin // R-type
                r_use_rs1=1; r_use_rs2=1; r_regwrite=1; r_alusrc=0; r_wbsel=2'd0;
                case ({funct7[5],funct3})
                    4'b0000: r_aluop=4'h0; // ADD
                    4'b1000: r_aluop=4'h1; // SUB
                    4'b0001: r_aluop=4'h2; // SLL
                    4'b0010: r_aluop=4'h3; // SLT
                    4'b0011: r_aluop=4'h4; // SLTU
                    4'b0100: r_aluop=4'h5; // XOR
                    4'b0101: r_aluop=4'h6; // SRL
                    4'b1101: r_aluop=4'h7; // SRA
                    4'b0110: r_aluop=4'h8; // OR
                    4'b0111: r_aluop=4'h9; // AND
                    default: r_aluop=4'h0;
                endcase
            end
            7'b0010011: begin // I-type ALU
                r_use_rs1=1; r_use_rs2=0; r_regwrite=1; r_alusrc=1; r_wbsel=2'd0;
                case (funct3)
                    3'b000: r_aluop=4'h0; // ADDI
                    3'b010: r_aluop=4'h3; // SLTI
                    3'b011: r_aluop=4'h4; // SLTIU
                    3'b100: r_aluop=4'h5; // XORI
                    3'b110: r_aluop=4'h8; // ORI
                    3'b111: r_aluop=4'h9; // ANDI
                    3'b001: r_aluop=4'h2; // SLLI
                    3'b101: r_aluop=(funct7[5]?4'h7:4'h6); // SRAI/SRLI
                    default: r_aluop=4'h0;
                endcase
            end
            7'b0000011: begin // LOAD
                r_use_rs1=1; r_regwrite=1; r_alusrc=1; r_memread=1; r_wbsel=2'd1; r_aluop=4'h0; // address = rs1 + imm
            end
            7'b0100011: begin // STORE
                r_use_rs1=1; r_use_rs2=1; r_alusrc=1; r_memwrite=1; r_aluop=4'h0;
            end
            7'b1100011: begin // BRANCH
                r_use_rs1=1; r_use_rs2=1; r_branch=1; r_aluop=4'h1; // use SUB for comparison baseline
            end
            7'b1101111: begin // JAL
                r_jal=1; r_regwrite=1; r_wbsel=2'd2;
            end
            7'b1100111: begin // JALR
                r_use_rs1=1; r_jalr=1; r_regwrite=1; r_wbsel=2'd2; r_alusrc=1; // target computed in EX
            end
            7'b0110111: begin // LUI
                r_regwrite=1; r_wbsel=2'd0; r_aluop=4'hA; r_alusrc=1; // ALU passes B
            end
            7'b0010111: begin // AUIPC
                r_regwrite=1; r_wbsel=2'd0; r_aluop=4'hB; r_alusrc=1; // ALU A=PC, B=imm
            end
            default: begin end
        endcase
    end

    assign use_rs1   = r_use_rs1;
    assign use_rs2   = r_use_rs2;
    assign is_branch = r_branch;
    assign is_jal    = r_jal;
    assign is_jalr   = r_jalr;
    assign memread   = r_memread;
    assign memwrite  = r_memwrite;
    assign regwrite  = r_regwrite;
    assign aluop     = r_aluop;
    assign alusrc    = r_alusrc;
    assign wb_sel    = r_wbsel;
endmodule

// ===================== ALU =====================
module alu(
    input  [3:0]  op,
    input  [31:0] a,
    input  [31:0] b,
    output [31:0] y
);
    reg [31:0] r;
    always @(*) begin
        case (op)
            4'h0: r = a + b;                       // ADD/ADDI/addr
            4'h1: r = a - b;                       // SUB (used for compare baseline)
            4'h2: r = a << b[4:0];                 // SLL/SLLI
            4'h3: r = ($signed(a) < $signed(b)) ? 32'd1 : 32'd0; // SLT/SLTI
            4'h4: r = (a < b) ? 32'd1 : 32'd0;     // SLTU/SLTIU
            4'h5: r = a ^ b;                       // XOR/XORI
            4'h6: r = a >> b[4:0];                 // SRL/SRLI
            4'h7: r = $signed(a) >>> b[4:0];       // SRA/SRAI
            4'h8: r = a | b;                       // OR/ORI
            4'h9: r = a & b;                       // AND/ANDI
            4'hA: r = b;                           // LUI pass-through
            4'hB: r = a + b;                       // AUIPC (A=PC)
            default: r = 32'b0;
        endcase
    end
    assign y = r;
endmodule

// ===================== Branch Unit =====================
module branch_unit(
    input  [2:0]  funct3,
    input  [31:0] rs1,
    input  [31:0] rs2,
    output        taken
);
    reg t;
    always @(*) begin
        case (funct3)
            3'b000: t = (rs1 == rs2);                    // BEQ
            3'b001: t = (rs1 != rs2);                    // BNE
            3'b100: t = ($signed(rs1) < $signed(rs2));   // BLT
            3'b101: t = !($signed(rs1) < $signed(rs2));  // BGE
            3'b110: t = (rs1 < rs2);                     // BLTU
            3'b111: t = !(rs1 < rs2);                    // BGEU
            default: t = 1'b0;
        endcase
    end
    assign taken = t;
endmodule

// ===================== Forwarding Unit =====================
module forwarding_unit(
    input  [4:0] idex_rs1,
    input  [4:0] idex_rs2,
    input  [4:0] exmem_rd,
    input        exmem_regwrite,
    input  [4:0] memwb_rd,
    input        memwb_regwrite,
    output [1:0] fwd_a,
    output [1:0] fwd_b
);
    reg [1:0] a,b;
    always @(*) begin
        // defaults (00: use ID/EX)
        a=2'd0; b=2'd0;
        // EX hazard
        if (exmem_regwrite && (exmem_rd!=5'd0) && (exmem_rd==idex_rs1)) a=2'd1;
        if (exmem_regwrite && (exmem_rd!=5'd0) && (exmem_rd==idex_rs2)) b=2'd1;
        // MEM hazard
        if (memwb_regwrite && (memwb_rd!=5'd0) && (memwb_rd==idex_rs1) && !(exmem_regwrite && (exmem_rd!=5'd0) && (exmem_rd==idex_rs1))) a=2'd2;
        if (memwb_regwrite && (memwb_rd!=5'd0) && (memwb_rd==idex_rs2) && !(exmem_regwrite && (exmem_rd!=5'd0) && (exmem_rd==idex_rs2))) b=2'd2;
    end
    assign fwd_a=a; assign fwd_b=b;
endmodule

// ===================== Store data aligner =====================
module store_align(
    input  [1:0]  addr,
    input  [2:0]  funct3,
    input  [31:0] rs2,
    output [3:0]  be,
    output [31:0] wdata
);
    reg [3:0]  r_be;
    reg [31:0] r_wdata;
    always @(*) begin
        case (funct3)
            3'b000: begin // SB
                case (addr)
                    2'b00: begin r_be=4'b0001; r_wdata={24'b0, rs2[7:0]}; end
                    2'b01: begin r_be=4'b0010; r_wdata={16'b0, rs2[7:0], 8'b0}; end
                    2'b10: begin r_be=4'b0100; r_wdata={8'b0, rs2[7:0], 16'b0}; end
                    default:begin r_be=4'b1000; r_wdata={rs2[7:0], 24'b0}; end
                endcase
            end
            3'b001: begin // SH
                if (addr[1]==1'b0) begin
                    r_be = 4'b0011; r_wdata = {16'b0, rs2[15:0]};
                end else begin
                    r_be = 4'b1100; r_wdata = {rs2[15:0], 16'b0};
                end
            end
            default: begin // SW
                r_be   = 4'b1111; r_wdata = rs2;
            end
        endcase
    end
    assign be = r_be; assign wdata = r_wdata;
endmodule

// ===================== Load data aligner =====================
module load_align(
    input  [1:0]  addr,
    input  [2:0]  funct3,
    input  [31:0] rdata,
    output [31:0] y
);
    reg [31:0] r;
    wire [7:0]  byte0 = rdata[7:0];
    wire [7:0]  byte1 = rdata[15:8];
    wire [7:0]  byte2 = rdata[23:16];
    wire [7:0]  byte3 = rdata[31:24];

    wire [15:0] half_lo = rdata[15:0];
    wire [15:0] half_hi = rdata[31:16];

    always @(*) begin
        case (funct3)
            3'b000: begin // LB
                case (addr)
                    2'b00: r = {{24{byte0[7]}}, byte0};
                    2'b01: r = {{24{byte1[7]}}, byte1};
                    2'b10: r = {{24{byte2[7]}}, byte2};
                    default: r = {{24{byte3[7]}}, byte3};
                endcase
            end
            3'b100: begin // LBU
                case (addr)
                    2'b00: r = {24'b0, byte0};
                    2'b01: r = {24'b0, byte1};
                    2'b10: r = {24'b0, byte2};
                    default: r = {24'b0, byte3};
                endcase
            end
            3'b001: begin // LH
                if (addr[1]==1'b0) r = {{16{half_lo[15]}}, half_lo};
                else               r = {{16{half_hi[15]}}, half_hi};
            end
            3'b101: begin // LHU
                if (addr[1]==1'b0) r = {16'b0, half_lo};
                else               r = {16'b0, half_hi};
            end
            default: begin // LW
                r = rdata;
            end
        endcase
    end
    assign y = r;
endmodule

// ===================== Simple test memory (optional) =====================
// This is a simple behavioral memory implementing both instruction and data
// ports. Useful for simulation-only bring up.
module simple_dualport_mem #(
    parameter MEM_BYTES = 4096
)(
    input         clk,
    input         rst,
    // instruction
    input         imem_req,
    input  [31:0] imem_addr,
    output reg    imem_rvalid,
    output [31:0] imem_rdata,
    // data
    input         dmem_req,
    input         dmem_we,
    input  [3:0]  dmem_be,
    input  [31:0] dmem_addr,
    input  [31:0] dmem_wdata,
    output        dmem_ready,
    output reg    dmem_rvalid,
    output [31:0] dmem_rdata
);
    reg [7:0] mem[0:MEM_BYTES-1];

    // Ready is always 1 in this simple model
    assign dmem_ready = 1'b1;

    // Instruction fetch (1-cycle latency)
    wire [31:0] iaddr = imem_addr;
    assign imem_rdata = {mem[iaddr+3], mem[iaddr+2], mem[iaddr+1], mem[iaddr+0]};

    // Data read (1-cycle latency)
    wire [31:0] daddr = dmem_addr;
    assign dmem_rdata = {mem[daddr+3], mem[daddr+2], mem[daddr+1], mem[daddr+0]};

    integer k;
    always @(posedge clk) begin
        if (rst) begin
            imem_rvalid <= 1'b0; dmem_rvalid <= 1'b0;
            // Optional: init memory to 0
            for (k=0; k<MEM_BYTES; k=k+1) mem[k] <= 8'h00;
        end else begin
            // reads are 1-cycle delayed valid
            imem_rvalid <= imem_req;
            dmem_rvalid <= dmem_req & ~dmem_we; // valid for loads

            // writes
            if (dmem_req && dmem_we) begin
                if (dmem_be[0]) mem[daddr+0] <= dmem_wdata[7:0];
                if (dmem_be[1]) mem[daddr+1] <= dmem_wdata[15:8];
                if (dmem_be[2]) mem[daddr+2] <= dmem_wdata[23:16];
                if (dmem_be[3]) mem[daddr+3] <= dmem_wdata[31:24];
            end
        end
    end
endmodule

// ===================== Example SoC wrapper (for quick sim) =====================
module rv32i_soc_example(
    input clk,
    input rst
);
    wire imem_req, imem_rvalid; wire [31:0] imem_addr, imem_rdata;
    wire dmem_req, dmem_we, dmem_ready, dmem_rvalid; wire [3:0] dmem_be; wire [31:0] dmem_addr, dmem_wdata, dmem_rdata;

    rv32i_core core(
        .clk(clk), .rst(rst),
        .imem_req(imem_req), .imem_addr(imem_addr), .imem_rvalid(imem_rvalid), .imem_rdata(imem_rdata),
        .dmem_req(dmem_req), .dmem_we(dmem_we), .dmem_be(dmem_be), .dmem_addr(dmem_addr), .dmem_wdata(dmem_wdata),
        .dmem_ready(dmem_ready), .dmem_rvalid(dmem_rvalid), .dmem_rdata(dmem_rdata)
    );

    simple_dualport_mem #(.MEM_BYTES(16384)) mem(
        .clk(clk), .rst(rst),
        .imem_req(imem_req), .imem_addr(imem_addr), .imem_rvalid(imem_rvalid), .imem_rdata(imem_rdata),
        .dmem_req(dmem_req), .dmem_we(dmem_we), .dmem_be(dmem_be), .dmem_addr(dmem_addr), .dmem_wdata(dmem_wdata),
        .dmem_ready(dmem_ready), .dmem_rvalid(dmem_rvalid), .dmem_rdata(dmem_rdata)
    );
endmodule


'''
    
    with open(filename, 'w') as f:
        f.write(enhanced_code)
    
    print(f"✅ Enhanced sample file created: {filename}")
    print(f"📏 Contains multiple rare region types for comprehensive testing")
    return filename


## saving the file 

In [4]:
create_enhanced_sample_sv_file(filename='RISCV.sv')

✅ Enhanced sample file created: RISCV.sv
📏 Contains multiple rare region types for comprehensive testing


'RISCV.sv'

# CHUNKING

## Cell 2: Core Chunk Generation Class Definition

In [5]:
# Cell 2: Core Chunk Generation Class Definition
# This class handles the sequential processing of SystemVerilog/Verilog files

class SequentialChunkGenerator:
    """
    Generates chunks from SystemVerilog/Verilog files in sequential order
    Maintains original line numbers while removing comments for clean content
    """
    
    def __init__(self):
        self.chunk_counter = 0
        self.chunks = {}
        self.file_content = ""
        self.file_lines = []
        self.current_line = 0
        
    def _detect_language(self, filepath: str) -> str:
        """Detect if file is SystemVerilog or Verilog based on extension"""
        ext = os.path.splitext(filepath)[1].lower()
        if ext == '.sv':
            return "SystemVerilog"
        elif ext == '.v':
            return "Verilog"
        else:
            return "Unknown"
    
    def _remove_comments(self, content: str) -> str:
        """
        Remove single-line (//) and multi-line (/* */) comments from code
        Preserves string literals and handles edge cases
        """
        # Remove single-line comments
        lines = content.split('\n')
        cleaned_lines = []
        
        for line in lines:
            # Find // but ignore if inside strings
            in_string = False
            escape_next = False
            comment_pos = -1
            
            for i, char in enumerate(line):
                if escape_next:
                    escape_next = False
                    continue
                    
                if char == '\\':
                    escape_next = True
                elif char == '"' and not escape_next:
                    in_string = not in_string
                elif not in_string and char == '/' and i + 1 < len(line) and line[i + 1] == '/':
                    comment_pos = i
                    break
            
            if comment_pos >= 0:
                line = line[:comment_pos].rstrip()
            
            cleaned_lines.append(line)
        
        content = '\n'.join(cleaned_lines)
        
        # Remove multi-line comments /* */
        content = re.sub(r'/\*.*?\*/', '', content, flags=re.DOTALL)
        
        return content
    
    def _clean_empty_lines(self, content: str) -> str:
        """Remove excessive empty lines while preserving structure"""
        lines = content.split('\n')
        cleaned_lines = []
        prev_empty = False
        
        for line in lines:
            is_empty = line.strip() == ''
            
            if not is_empty:
                cleaned_lines.append(line)
                prev_empty = False
            elif not prev_empty:
                cleaned_lines.append(line)
                prev_empty = True
        
        return '\n'.join(cleaned_lines)

print("✅ SequentialChunkGenerator class defined")
print("🔧 Core functionality: comment removal, language detection, content cleaning")

✅ SequentialChunkGenerator class defined
🔧 Core functionality: comment removal, language detection, content cleaning


## Cell 3: Module Detection Methods

In [6]:
# Cell 3: Module Detection and Boundary Identification
# These methods identify SystemVerilog/Verilog constructs and their boundaries

class SequentialChunkGenerator(SequentialChunkGenerator):
    """Extended with module detection capabilities"""
    
    def _find_module_boundaries(self, content: str) -> List[Dict]:
        """
        Find all module boundaries in sequential order
        Returns list of modules with start/end positions
        """
        modules = []
        lines = content.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            # Check for module declaration
            module_match = re.match(r'module\s+(\w+)', line)
            if module_match:
                module_name = module_match.group(1)
                start_line = i + 1  # 1-based line numbering
                
                # Find corresponding endmodule
                end_line = self._find_endmodule(lines, i)
                
                if end_line:
                    modules.append({
                        'name': module_name,
                        'start_line': start_line,
                        'end_line': end_line,
                        'type': 'module'
                    })
                    i = end_line  # Skip to after this module
                else:
                    i += 1
            else:
                i += 1
        
        return modules
    
    def _find_endmodule(self, lines: List[str], start_idx: int) -> Optional[int]:
        """Find the corresponding endmodule for a module starting at start_idx"""
        module_depth = 1
        
        for i in range(start_idx + 1, len(lines)):
            line = lines[i].strip()
            
            if re.match(r'module\s+\w+', line):
                module_depth += 1
            elif line.startswith('endmodule'):
                module_depth -= 1
                if module_depth == 0:
                    return i + 1  # 1-based line numbering
        
        return None
    
    def _identify_miscellaneous_chunks(self, content: str, modules: List[Dict]) -> List[Dict]:
        """
        Identify non-module chunks (timescale, includes, etc.)
        These are processed in order between modules
        """
        misc_chunks = []
        lines = content.split('\n')
        
        # Find gaps between modules for miscellaneous content
        covered_lines = set()
        for module in modules:
            for line_num in range(module['start_line'], module['end_line'] + 1):
                covered_lines.add(line_num)
        
        # Group uncovered lines into chunks
        current_chunk_start = None
        
        for line_num in range(1, len(lines) + 1):
            if line_num not in covered_lines:
                line_content = lines[line_num - 1].strip()
                
                # Skip completely empty lines
                if line_content:
                    if current_chunk_start is None:
                        current_chunk_start = line_num
            else:
                # End current misc chunk if we hit a module
                if current_chunk_start is not None:
                    misc_chunks.append({
                        'name': f'misc_{current_chunk_start}',
                        'start_line': current_chunk_start,
                        'end_line': line_num - 1,
                        'type': 'miscellaneous'
                    })
                    current_chunk_start = None
        
        # Handle final misc chunk
        if current_chunk_start is not None:
            misc_chunks.append({
                'name': f'misc_{current_chunk_start}',
                'start_line': current_chunk_start,
                'end_line': len(lines),
                'type': 'miscellaneous'
            })
        
        return misc_chunks

print("✅ Module detection methods added")
print("🔍 Capabilities: module boundaries, endmodule detection, miscellaneous chunks")

✅ Module detection methods added
🔍 Capabilities: module boundaries, endmodule detection, miscellaneous chunks


## Cell 4: Chunk Creation and Processing

In [7]:
# Cell 4: Chunk Creation and Content Processing (Updated)
# Handles chunk creation with 150-line limit for sub-chunking

class SequentialChunkGenerator(SequentialChunkGenerator):
    """Extended with chunk creation and processing capabilities"""
    
    def __init__(self):
        super().__init__()
        self.remove_comments_enabled = True  # Default: remove comments
    
    def _create_chunk(self, construct: Dict, content: str) -> Dict:
        """
        Create a chunk from a construct (module or miscellaneous)
        Extracts content, removes comments, and calculates metadata
        """
        self.chunk_counter += 1
        chunk_id = f"chunk_{self.chunk_counter:03d}"
        
        # Extract content for this chunk
        lines = content.split('\n')
        start_idx = construct['start_line'] - 1  # Convert to 0-based
        end_idx = construct['end_line'] - 1      # Convert to 0-based
        
        chunk_lines = lines[start_idx:end_idx + 1]
        raw_content = '\n'.join(chunk_lines)
        
        # Clean content (remove comments if enabled)
        if self.remove_comments_enabled:
            cleaned_content = self._remove_comments(raw_content)
            cleaned_content = self._clean_empty_lines(cleaned_content)
        else:
            cleaned_content = raw_content
        
        # Calculate actual content lines (non-empty after cleaning)
        content_lines = [line for line in cleaned_content.split('\n') if line.strip()]
        
        chunk = {
            "chunk_id": chunk_id,
            "start_line": construct['start_line'],
            "end_line": construct['end_line'],
            "content": cleaned_content,
            "parent_module": None,
            "parent_construct": construct.get('name', None),
            "is_sub_chunk": False,
            "parent_chunk": None,
            "line_count": len(content_lines),
            "construct_type": construct['type']
        }
        
        return chunk_id, chunk
    
    def _should_split_module(self, chunk: Dict) -> bool:
        """
        Determine if a module should be split into sub-chunks
        Split only if module is larger than 150 lines
        """
        return chunk['construct_type'] == 'module' and chunk['line_count'] > 150
    
    def _create_sub_chunks(self, parent_chunk_id: str, parent_chunk: Dict) -> Dict:
        """
        Split large modules into smaller sub-chunks (a, b, c, etc.)
        Maintains parent-child relationships
        """
        sub_chunks = {}
        content = parent_chunk['content']
        lines = content.split('\n')
        content_lines = [line for line in lines if line.strip()]
        
        # Calculate how many sub-chunks we need
        total_content_lines = len(content_lines)
        chunks_needed = max(2, (total_content_lines + 149) // 150)  # Ceiling division
        
        print(f"     📦 Splitting {total_content_lines} lines into {chunks_needed} sub-chunks")
        
        # Find good split points
        split_points = []
        chunk_size = len(lines) // chunks_needed
        
        for i in range(1, chunks_needed):
            target_split = i * chunk_size
            actual_split = self._find_good_split_point(lines, target_split)
            split_points.append(actual_split)
        
        # Create sub-chunks
        sub_chunk_labels = ['a', 'b', 'c', 'd', 'e']  # Support up to 5 sub-chunks
        prev_split = 0
        
        for i in range(chunks_needed):
            sub_chunk_id = f"{parent_chunk_id}_{sub_chunk_labels[i]}"
            
            # Determine end point
            if i == chunks_needed - 1:
                # Last chunk gets everything remaining
                end_split = len(lines)
            else:
                end_split = split_points[i]
            
            # Extract content for this sub-chunk
            sub_chunk_lines = lines[prev_split:end_split]
            sub_chunk_content = '\n'.join(sub_chunk_lines)
            sub_content_lines = [line for line in sub_chunk_lines if line.strip()]
            
            sub_chunk = {
                "chunk_id": sub_chunk_id,
                "start_line": parent_chunk['start_line'] + prev_split,
                "end_line": parent_chunk['start_line'] + end_split - 1,
                "content": sub_chunk_content,
                "parent_module": None,
                "parent_construct": parent_chunk['parent_construct'],
                "is_sub_chunk": True,
                "parent_chunk": parent_chunk_id,
                "line_count": len(sub_content_lines),
                "construct_type": parent_chunk['construct_type']
            }
            
            sub_chunks[sub_chunk_id] = sub_chunk
            print(f"       └─ {sub_chunk_id}: {len(sub_content_lines)} content lines")
            
            prev_split = end_split
        
        return sub_chunks
    
    def _find_good_split_point(self, lines: List[str], target_line: int) -> int:
        """Find a logical place to split the module (after semicolon, end, etc.)"""
        search_range = 20  # Look within 20 lines of target
        
        # Preferred split patterns (in order of preference)
        good_split_patterns = [
            r'^\s*end\s*$',                    # 'end' statements
            r'^\s*endmodule\s*$',              # 'endmodule' 
            r';\s*$',                          # Lines ending with semicolon
            r'^\s*\)\s*;\s*$',                 # Closing parenthesis with semicolon
            r'^\s*end\w+\s*$',                 # end something (endcase, etc.)
        ]
        
        # First try to find good patterns near target
        for offset in range(search_range):
            for line_idx in [target_line + offset, target_line - offset]:
                if 0 <= line_idx < len(lines):
                    line = lines[line_idx].strip()
                    
                    for pattern in good_split_patterns:
                        if re.search(pattern, line):
                            return line_idx + 1
        
        # Fallback: find any non-empty line near target
        for offset in range(search_range):
            for line_idx in [target_line + offset, target_line - offset]:
                if 0 <= line_idx < len(lines):
                    if lines[line_idx].strip():
                        return line_idx + 1
        
        # Final fallback to target line
        return max(1, min(target_line, len(lines)))

print("✅ Chunk creation methods updated")
print("🎯 Features: 150-line limit, smart sub-chunking, optional comment removal")

✅ Chunk creation methods updated
🎯 Features: 150-line limit, smart sub-chunking, optional comment removal


## Cell 5: Main Processing Pipeline

In [8]:
# Cell 5: Main Processing Pipeline
# The core method that orchestrates the entire chunk generation process

class SequentialChunkGenerator(SequentialChunkGenerator):
    """Complete chunk generator with main processing pipeline"""
    
    def generate_chunks(self, filepath: str) -> Dict:
        """
        Main method to generate chunks from a SystemVerilog/Verilog file
        Processes file sequentially to maintain order
        
        Args:
            filepath: Path to the .sv or .v file
            
        Returns:
            Dictionary containing file metadata, chunks, and statistics
        """
        print(f"🚀 Starting chunk generation for: {os.path.basename(filepath)}")
        
        # Reset state for new file
        self.chunk_counter = 0
        self.chunks = {}
        
        # Read file
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                self.file_content = f.read()
        except Exception as e:
            raise Exception(f"Failed to read file {filepath}: {e}")
        
        self.file_lines = self.file_content.split('\n')
        
        # Generate file metadata
        file_stats = os.stat(filepath)
        file_metadata = {
            "filename": os.path.basename(filepath),
            "filepath": filepath,
            "total_lines": len(self.file_lines),
            "language": self._detect_language(filepath),
            "file_size_bytes": file_stats.st_size,
            "created_at": datetime.now().isoformat()
        }
        
        print(f"📁 File info: {file_metadata['total_lines']} lines, {file_metadata['language']}")
        
        # Find all constructs in sequential order
        modules = self._find_module_boundaries(self.file_content)
        misc_chunks = self._identify_miscellaneous_chunks(self.file_content, modules)
        
        # Combine and sort all constructs by start_line
        all_constructs = modules + misc_chunks
        all_constructs.sort(key=lambda x: x['start_line'])
        
        print(f"🔍 Found {len(modules)} modules and {len(misc_chunks)} misc sections")
        
        # Process constructs sequentially
        chunk_sequence = []
        sub_chunks_created = 0
        
        for construct in all_constructs:
            print(f"   Processing {construct['type']}: {construct.get('name', 'misc')} (lines {construct['start_line']}-{construct['end_line']})")
            
            # Create main chunk
            chunk_id, chunk = self._create_chunk(construct, self.file_content)
            self.chunks[chunk_id] = chunk
            chunk_sequence.append(chunk_id)
            
            # Check if we need to create sub-chunks
            if self._should_split_module(chunk):
                print(f"     📦 Large module ({chunk['line_count']} lines) - creating sub-chunks")
                
                # Remove the parent chunk from main chunks
                del self.chunks[chunk_id]
                chunk_sequence.remove(chunk_id)
                
                # Create sub-chunks
                sub_chunks = self._create_sub_chunks(chunk_id, chunk)
                
                # Add sub-chunks to main collection
                for sub_chunk_id, sub_chunk in sub_chunks.items():
                    self.chunks[sub_chunk_id] = sub_chunk
                    chunk_sequence.append(sub_chunk_id)
                
                sub_chunks_created += len(sub_chunks)
        
        # Generate statistics
        total_content_lines = sum(chunk['line_count'] for chunk in self.chunks.values())
        
        chunking_stats = {
            "total_chunks": len(self.chunks),
            "sub_chunks": sub_chunks_created,
            "average_chunk_size": total_content_lines / len(self.chunks) if self.chunks else 0,
            "constructs_preserved": len(modules),
            "comments_removed": True
        }
        
        print(f"✅ Generated {len(self.chunks)} chunks ({sub_chunks_created} sub-chunks)")
        print(f"📊 Average chunk size: {chunking_stats['average_chunk_size']:.1f} lines")
        
        # Return complete chunk data
        return {
            "file_metadata": file_metadata,
            "chunks": self.chunks,
            "chunk_sequence": chunk_sequence,
            "chunking_stats": chunking_stats
        }

print("✅ Main processing pipeline complete")
print("🎯 Ready to process SystemVerilog/Verilog files sequentially")

✅ Main processing pipeline complete
🎯 Ready to process SystemVerilog/Verilog files sequentially


## Cell 6: Utility Functions

In [9]:
# Cell 6: Utility Functions
# Helper functions for file operations and chunk validation

def save_chunks_to_file(chunks_data: Dict, output_path: str) -> str:
    """
    Save chunk data to JSON file with proper formatting
    
    Args:
        chunks_data: Dictionary containing chunk information
        output_path: Path where to save the JSON file
        
    Returns:
        Path to saved file
    """
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(chunks_data, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Chunks saved to: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"❌ Failed to save chunks: {e}")
        raise

def generate_chunks_for_file(input_filepath: str, output_dir: str = None) -> str:
    """
    Complete workflow to generate chunks for a single file
    
    Args:
        input_filepath: Path to input .sv or .v file
        output_dir: Directory to save output (default: same as input)
        
    Returns:
        Path to generated chunks file
    """
    # Determine output path
    if output_dir is None:
        output_dir = os.path.dirname(input_filepath)
    
    base_name = os.path.splitext(os.path.basename(input_filepath))[0]
    output_path = os.path.join(output_dir, f"{base_name}_chunks.json")
    
    # Generate chunks
    generator = SequentialChunkGenerator()
    chunks_data = generator.generate_chunks(input_filepath)
    
    # Save to file
    save_chunks_to_file(chunks_data, output_path)
    
    return output_path

def validate_chunks(chunks_data: Dict) -> bool:
    """
    Basic validation of chunk data structure
    
    Args:
        chunks_data: Chunk data dictionary
        
    Returns:
        True if valid, False otherwise
    """
    try:
        # Check required top-level keys
        required_keys = ['file_metadata', 'chunks', 'chunk_sequence', 'chunking_stats']
        for key in required_keys:
            if key not in chunks_data:
                print(f"❌ Missing required key: {key}")
                return False
        
        # Check chunk sequence matches chunks
        chunk_ids = set(chunks_data['chunks'].keys())
        sequence_ids = set(chunks_data['chunk_sequence'])
        
        if chunk_ids != sequence_ids:
            print(f"❌ Chunk sequence mismatch")
            print(f"   Chunks: {sorted(chunk_ids)}")
            print(f"   Sequence: {sorted(sequence_ids)}")
            return False
        
        # Check chunk line numbers are sequential
        chunks = chunks_data['chunks']
        sequence = chunks_data['chunk_sequence']
        
        prev_end_line = 0
        for chunk_id in sequence:
            chunk = chunks[chunk_id]
            if chunk['start_line'] <= prev_end_line:
                # Allow some overlap for sub-chunks
                if not chunk.get('is_sub_chunk', False):
                    print(f"❌ Chunk {chunk_id} has invalid line ordering")
                    return False
            prev_end_line = chunk['end_line']
        
        print("✅ Chunk validation passed")
        return True
        
    except Exception as e:
        print(f"❌ Validation error: {e}")
        return False

print("✅ Utility functions ready")
print("🛠️ Features: file saving, workflow automation, validation")

✅ Utility functions ready
🛠️ Features: file saving, workflow automation, validation


## Cell 8: Simplified Interactive Chunk Generator

In [10]:
# Cell 8: Simplified Interactive Chunk Generator (Updated)
# Simple interface with 150-line sub-chunking and comment removal option

def simplified_chunk_generator():
    """
    Simplified chunk generator with 150-line limit
    Automatically saves with filename_chunks.json naming
    Shows first chunk as preview
    """
    print("🎯 Simplified Chunk Generator (150-line limit)")
    print("=" * 55)
    
    # Get input file path
    input_file = input("📁 Enter file path: ").strip()
    
    # Check if file exists
    if not os.path.exists(input_file):
        print(f"❌ File not found: {input_file}")
        return None
    
    # Ask about comment removal (default True)
    remove_comments_input = input("🧹 Remove comments? (Y/n, default=Y): ").strip().lower()
    remove_comments = remove_comments_input != 'n'
    
    print(f"   Comment removal: {'✅ Enabled' if remove_comments else '❌ Disabled'}")
    print(f"   Sub-chunk limit: 150 lines per chunk")
    
    # Generate output filename automatically
    base_name = os.path.splitext(os.path.basename(input_file))[0]
    output_dir = "/kaggle/working"
    output_file = os.path.join(output_dir, f"{base_name}_chunks.json")
    
    # Process file
    try:
        print(f"\n🚀 Processing: {os.path.basename(input_file)}")
        print(f"💾 Output will be saved as: {os.path.basename(output_file)}")
        
        # Create generator with comment removal option
        generator = SequentialChunkGenerator()
        generator.remove_comments_enabled = remove_comments
        
        chunks_data = generator.generate_chunks(input_file)
        
        # Save chunks
        save_chunks_to_file(chunks_data, output_file)
        
        # Display summary
        print(f"\n📊 Processing Complete!")
        print(f"   ✅ File: {chunks_data['file_metadata']['filename']}")
        print(f"   ✅ Language: {chunks_data['file_metadata']['language']}")
        print(f"   ✅ Total lines: {chunks_data['file_metadata']['total_lines']}")
        print(f"   ✅ Total chunks: {len(chunks_data['chunks'])}")
        print(f"   ✅ Sub-chunks: {chunks_data['chunking_stats']['sub_chunks']}")
        print(f"   ✅ Output saved: {output_file}")
        
        # Show chunk breakdown
        print(f"\n📋 Chunk Breakdown:")
        print("-" * 40)
        for i, chunk_id in enumerate(chunks_data['chunk_sequence'], 1):
            chunk = chunks_data['chunks'][chunk_id]
            sub_indicator = " (sub)" if chunk.get('is_sub_chunk', False) else ""
            print(f"   {i:2d}. {chunk_id}{sub_indicator}: lines {chunk['start_line']}-{chunk['end_line']} ({chunk['line_count']} lines)")
        
        # Show first chunk as preview
        print(f"\n📋 First Chunk Preview:")
        print("=" * 50)
        
        first_chunk_id = chunks_data['chunk_sequence'][0]
        first_chunk = chunks_data['chunks'][first_chunk_id]
        
        print(f"Chunk ID: {first_chunk_id}")
        print(f"Lines: {first_chunk['start_line']}-{first_chunk['end_line']} ({first_chunk['line_count']} content lines)")
        print(f"Type: {first_chunk['construct_type']}")
        print(f"Construct: {first_chunk['parent_construct'] or 'N/A'}")
        print(f"Sub-chunk: {'Yes' if first_chunk.get('is_sub_chunk', False) else 'No'}")
        print("\nContent (first 15 lines):")
        print("-" * 30)
        
        content_lines = first_chunk['content'].split('\n')
        for i, line in enumerate(content_lines[:15], 1):
            print(f"{i:2d}: {line}")
        
        if len(content_lines) > 15:
            print(f"... ({len(content_lines) - 15} more lines)")
        
        print("-" * 30)
        
        # Quick validation
        is_valid = validate_chunks(chunks_data)
        print(f"\n✅ Validation: {'PASSED' if is_valid else 'FAILED'}")
        
        return chunks_data
        
    except Exception as e:
        print(f"❌ Error during processing: {e}")
        return None

# Ready to use
print("🎯 Ready to run: simplified_chunk_generator()")
print("   • 150-line limit for sub-chunking")
print("   • Module-based chunking strategy")
print("   • Optional comment removal (default: Yes)")
print("   • Sequential ordering guaranteed")
print("\n🚀 Run: simplified_chunk_generator()")

🎯 Ready to run: simplified_chunk_generator()
   • 150-line limit for sub-chunking
   • Module-based chunking strategy
   • Optional comment removal (default: Yes)
   • Sequential ordering guaranteed

🚀 Run: simplified_chunk_generator()


## Usage

In [11]:
# Replace with your actual file path
input_file = "/kaggle/working/RISCV.sv"

# This creates filename_chunks.json automatically
output_file = generate_chunks_for_file(input_file)

print(f"✅ Chunks saved to: {output_file}")

🚀 Starting chunk generation for: RISCV.sv
📁 File info: 709 lines, SystemVerilog
🔍 Found 11 modules and 11 misc sections
   Processing miscellaneous: misc_1 (lines 1-37)
   Processing module: rv32i_core (lines 38-328)
     📦 Large module (227 lines) - creating sub-chunks
     📦 Splitting 227 lines into 2 sub-chunks
       └─ chunk_002_a: 114 content lines
       └─ chunk_002_b: 113 content lines
   Processing miscellaneous: misc_330 (lines 330-330)
   Processing module: regfile (lines 331-352)
   Processing miscellaneous: misc_354 (lines 354-354)
   Processing module: immgen (lines 355-372)
   Processing miscellaneous: misc_374 (lines 374-374)
   Processing module: control (lines 375-465)
   Processing miscellaneous: misc_467 (lines 467-467)
   Processing module: alu (lines 468-493)
   Processing miscellaneous: misc_495 (lines 495-495)
   Processing module: branch_unit (lines 496-515)
   Processing miscellaneous: misc_517 (lines 517-517)
   Processing module: forwarding_unit (lines 518-

# Rare region finding

## Cell 9: Rare Region Detection with Chunk Mapping

In [12]:
# Cell 9: Simplified Rare Region Detection with Exact Code Blocks
# Clean prompt + robust multi-line code matching

import json
import pandas as pd
import re
from datetime import datetime

def interactive_analysis(chunks_file):
    """
    Interactive analysis to create rare regions JSON with chunk location mapping.
    Simple prompt: Get EXACT VERBATIM code blocks from GPT.
    """
    # Load chunks
    with open(chunks_file, 'r') as f:
        chunks_data = json.load(f)
    
    all_rare_regions = []
    chunks_with_regions = 0
    total_chunks = len(chunks_data['chunks'])
    
    print(f"🚀 Starting analysis of {total_chunks} chunks...")
    print("=" * 60)
    
    for chunk_id, chunk in chunks_data['chunks'].items():
        print(f"\n🧩 CHUNK: {chunk_id}")
        print("🏗️ TYPE:", chunk.get('construct_type', 'unknown'), "| 📦 MODULE:", chunk.get('parent_module', 'None'))
        
        print("\nTASK: Analyze this SystemVerilog code and identify RARE EXECUTION REGIONS.")
        print("\nLOOK FOR any code patterns that execute INFREQUENTLY during normal operation:")
        print("• Conditional blocks that trigger only under specific/uncommon conditions")
        print("• Error handling paths and exception cases")
        print("• Initialization, reset, or startup sequences")
        print("• Debug, test, or configuration modes")
        print("• Complex loops or computationally expensive operations")
        print("• Security checks, privilege validations, or audit trails")
        print("• Cache miss handlers, memory fault recovery")
        print("• Calibration routines, self-test procedures")
        print("• Interrupt handlers for rare events")
        print("• Fallback mechanisms or recovery procedures")
        print("• Any 'if' conditions that rarely evaluate to true")
        print("• Code paths triggered only during specific system states")
        
        print("\nCODE TO ANALYZE:")
        print("─" * 40)
        print(chunk['content'])
        print("─" * 40)
        
        # SIMPLE AND CLEAR PROMPT
        print("\n" + "=" * 60)
        print("📋 CODE SNIPPET REQUIREMENTS:")
        print("=" * 60)
        print("• Provide the EXACT multi-line code block as it appears in the chunk")
        print("• Copy it VERBATIM - no paraphrasing, no abbreviation, no ellipsis (...)")
        print("• Include ALL lines that constitute the rare region")
        print("\n📋 REQUIRED JSON OUTPUT FORMAT:")
        print('[')
        print('  {')
        print('    "rare_region_type": "HAZARD_DETECTION",')
        print('    "code_snippet": "wire load_use_hazard = idex_memread & idex_valid & ifid_valid & (\\n    (id_use_rs1 && (idex_rd != 5\'d0) && (idex_rd == id_rs1)) ||\\n    (id_use_rs2 && (idex_rd != 5\'d0) && (idex_rd == id_rs2))\\n);",')
        print('    "activation_trigger": "Load-use hazard condition",')
        print('    "confidence_score": 0.9,')
        print('    "description": "Pipeline hazard detection requiring stall"')
        print('  }')
        print(']')
        print("\n💡 If NO rare regions found, output: []")
        print("=" * 60)
        
        # Collect GPT response
        print(f"\n📋 Paste GPT response for {chunk_id} (type 'DONE' on new line when finished):")
        gpt_response = []
        while True:
            line = input()
            if line.strip() == 'DONE':
                break
            gpt_response.append(line)
        
        # Process response and find locations in chunk
        response_text = '\n'.join(gpt_response)
        chunk_regions = process_gpt_response_with_location(response_text, chunk_id, chunk)
        
        if chunk_regions:
            all_rare_regions.extend(chunk_regions)
            chunks_with_regions += 1
            print(f"✅ Found {len(chunk_regions)} rare regions in {chunk_id}")
        else:
            print(f"ℹ️ No rare regions found in {chunk_id}")
    
    # Create final JSON
    output_file = chunks_file.replace('_chunks.json', '_rare_regions.json')
    result = {
        "file_metadata": {
            "source_file": chunks_data['file_metadata']['filename'],
            "total_chunks_analyzed": total_chunks,
            "chunks_with_rare_regions": chunks_with_regions,
            "total_rare_regions": len(all_rare_regions),
            "analysis_timestamp": datetime.now().isoformat()
        },
        "rare_regions": all_rare_regions
    }
    
    with open(output_file, 'w') as f:
        json.dump(result, f, indent=2)
    
    print(f"\n🎯 Analysis Complete!")
    print(f"📄 JSON saved to: {output_file}")
    print(f"📊 Total rare regions: {len(all_rare_regions)}")
    return output_file


def find_multiline_code_in_chunk(code_snippet: str, chunk_content: str) -> dict:
    """
    Find multi-line code snippets in chunk using smart matching strategies.
    
    Strategies (in order):
    1. First-line exact match (use first line as anchor)
    2. Full exact match (for single-line or perfect copies)
    3. Normalized whitespace match (handles spacing differences)
    4. Keyword-based fuzzy match (fallback)
    """
    chunk_lines = chunk_content.split('\n')
    snippet_clean = code_snippet.strip()
    
    # Split snippet into lines
    snippet_lines = snippet_clean.split('\n')
    num_snippet_lines = len(snippet_lines)
    
    # STRATEGY 1: First-line anchor match (most reliable for multi-line)
    if num_snippet_lines > 0:
        first_line = snippet_lines[0].strip()
        
        for i, chunk_line in enumerate(chunk_lines):
            chunk_line_clean = chunk_line.strip()
            
            # Check if first line matches
            if first_line == chunk_line_clean or first_line in chunk_line_clean:
                return {
                    "line_in_chunk": i + 1,
                    "matched_line": chunk_line_clean,
                    "match_type": "first_line_exact",
                    "line_range": f"{i + 1}" if num_snippet_lines == 1 else f"{i + 1}-{i + num_snippet_lines}"
                }
    
    # STRATEGY 2: Full exact match (single line or perfect multi-line)
    for i, chunk_line in enumerate(chunk_lines):
        if snippet_clean in chunk_line.strip():
            return {
                "line_in_chunk": i + 1,
                "matched_line": chunk_line.strip(),
                "match_type": "full_exact",
                "line_range": str(i + 1)
            }
    
    # STRATEGY 3: Normalized whitespace match
    # Remove all whitespace and compare
    snippet_normalized = re.sub(r'\s+', '', snippet_clean.lower())
    
    # Try sliding window for multi-line matches
    for i in range(len(chunk_lines)):
        # Try windows of different sizes
        for window_size in range(1, min(num_snippet_lines + 3, len(chunk_lines) - i + 1)):
            window_lines = chunk_lines[i:i + window_size]
            window_text = '\n'.join(window_lines)
            window_normalized = re.sub(r'\s+', '', window_text.lower())
            
            # Check if normalized snippet is in normalized window
            if snippet_normalized in window_normalized:
                return {
                    "line_in_chunk": i + 1,
                    "matched_line": chunk_lines[i].strip(),
                    "match_type": "normalized_whitespace",
                    "line_range": f"{i + 1}" if window_size == 1 else f"{i + 1}-{i + window_size}"
                }
    
    # STRATEGY 4: Keyword-based fuzzy match (last resort)
    # Extract important keywords from first line
    first_line = snippet_lines[0] if snippet_lines else snippet_clean
    keywords = extract_keywords(first_line)
    
    if len(keywords) >= 2:
        best_match_line = None
        best_match_score = 0
        
        for i, chunk_line in enumerate(chunk_lines):
            line_keywords = set(re.findall(r'\w+', chunk_line.lower()))
            common_keywords = keywords & line_keywords
            
            # Calculate match score
            score = len(common_keywords) / len(keywords) if keywords else 0
            
            if score > best_match_score and score >= 0.5:  # 50% threshold
                best_match_score = score
                best_match_line = i
        
        if best_match_line is not None:
            return {
                "line_in_chunk": best_match_line + 1,
                "matched_line": chunk_lines[best_match_line].strip(),
                "match_type": f"fuzzy_keywords_{best_match_score:.2f}",
                "line_range": f"{best_match_line + 1}" if num_snippet_lines == 1 else f"{best_match_line + 1}-{best_match_line + num_snippet_lines}"
            }
    
    # No match found
    return None


def extract_keywords(code_line: str) -> set:
    """
    Extract meaningful keywords from code, filtering common noise.
    """
    # Verilog/SystemVerilog noise words
    noise = {
        'wire', 'reg', 'input', 'output', 'logic', 'assign', 'begin', 'end',
        'if', 'else', 'case', 'for', 'while', 'module', 'endmodule',
        'always', 'initial', 'posedge', 'negedge'
    }
    
    words = re.findall(r'\w+', code_line.lower())
    return set(w for w in words if w not in noise and len(w) > 1)


def process_gpt_response_with_location(response_text, chunk_id, chunk):
    """
    Process GPT JSON response and find code locations within chunk.
    """
    lines = response_text.strip()
    regions = []
    
    # Parse JSON
    try:
        if lines.startswith('[') and lines.endswith(']'):
            json_data = json.loads(lines)
            
            for region in json_data:
                code_snippet = region.get('code_snippet', '')
                
                # Find location using multi-strategy matching
                location_info = find_multiline_code_in_chunk(code_snippet, chunk['content'])
                
                # Build region entry
                enhanced_region = {
                    "chunk_id": chunk_id,
                    "rare_region_type": region.get('rare_region_type', 'UNKNOWN'),
                    "rare_region_code": code_snippet,
                    "activation_trigger": region.get('activation_trigger', 'Unknown trigger'),
                    "confidence_score": region.get('confidence_score', 0.5),
                    "description": region.get('description', 'No description provided')
                }
                
                # Add location info
                if location_info:
                    enhanced_region.update({
                        "line_in_chunk": location_info["line_in_chunk"],
                        "absolute_line": chunk['start_line'] + location_info["line_in_chunk"] - 1,
                        "matched_line_content": location_info["matched_line"],
                        "match_type": location_info["match_type"],
                        "line_range": location_info["line_range"]
                    })
                    
                    preview = code_snippet.replace('\n', ' ')[:50]
                    print(f"   📍 Found at line {location_info['line_range']} ({location_info['match_type']})")
                else:
                    enhanced_region.update({
                        "line_in_chunk": None,
                        "absolute_line": None,
                        "matched_line_content": None,
                        "match_type": "not_found",
                        "line_range": None
                    })
                    preview = code_snippet.replace('\n', ' ')[:50]
                    print(f"   ⚠️ Could not locate: {preview}...")
                
                regions.append(enhanced_region)
            
            return regions
            
    except json.JSONDecodeError as e:
        print(f"   ❌ JSON parsing failed: {e}")
    
    return regions


def view_results(json_file):
    """View results in readable format"""
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    print("📊 RARE REGIONS ANALYSIS RESULTS")
    print("=" * 50)
    print(f"📁 Source: {data['file_metadata']['source_file']}")
    print(f"📈 Total Chunks: {data['file_metadata']['total_chunks_analyzed']}")
    print(f"🎯 Chunks with Regions: {data['file_metadata']['chunks_with_rare_regions']}")
    print(f"🔍 Total Rare Regions: {data['file_metadata']['total_rare_regions']}")
    
    print("\n🧩 RARE REGIONS BY CHUNK:")
    print("-" * 50)
    
    current_chunk = None
    for region in data['rare_regions']:
        if region['chunk_id'] != current_chunk:
            current_chunk = region['chunk_id']
            print(f"\n📦 {current_chunk}:")
        
        code_preview = region['rare_region_code'].replace('\n', ' ')[:50] + '...'
        line_info = region.get('line_range', 'N/A')
        
        print(f"  • {region['rare_region_type']}")
        print(f"    📍 Lines: {line_info} | Match: {region.get('match_type', 'N/A')}")
        print(f"    Code: {code_preview}")


def json_to_csv(json_file):
    """Convert JSON to CSV"""
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data['rare_regions'])
    csv_file = json_file.replace('.json', '.csv')
    df.to_csv(csv_file, index=False)
    print(f"✅ CSV saved to: {csv_file}")
    return csv_file


# Ready to use
print("🚀 SIMPLIFIED RARE REGION ANALYSIS READY!")
print("=" * 50)
print("\n📝 Usage:")
print("  interactive_analysis('/kaggle/working/filename_chunks.json')")
print("  view_results('/kaggle/working/filename_rare_regions.json')")
print("\n✨ Features:")
print("  • Clean, simple prompt - just asks for EXACT VERBATIM code")
print("  • 4-strategy matching: exact → first-line → normalized → fuzzy")
print("  • Handles single-line and multi-line code blocks")
print("  • Reports line ranges for multi-line matches")
print("\n🎯 Ready to start!")

🚀 SIMPLIFIED RARE REGION ANALYSIS READY!

📝 Usage:
  interactive_analysis('/kaggle/working/filename_chunks.json')
  view_results('/kaggle/working/filename_rare_regions.json')

✨ Features:
  • Clean, simple prompt - just asks for EXACT VERBATIM code
  • 4-strategy matching: exact → first-line → normalized → fuzzy
  • Handles single-line and multi-line code blocks
  • Reports line ranges for multi-line matches

🎯 Ready to start!


## Cell 13: Smart Pipeline Wrapper

In [13]:
# Cell 13: Smart Pipeline Wrapper
# Uses existing functions with flexibility for different starting points

def analyze_file_complete(file_path: str, remove_comments: bool = True):
    """
    Complete pipeline: file → chunks → rare regions
    
    Args:
        file_path: Path to .sv or .v file
        remove_comments: Whether to remove comments (default: True)
    """
    print("🚀 COMPLETE PIPELINE: File → Chunks → Rare Regions")
    print("=" * 55)
    
    # Step 1: Generate chunks
    print("\n📦 STEP 1: Generating chunks...")
    generator = SequentialChunkGenerator()
    generator.remove_comments_enabled = remove_comments
    
    chunks_data = generator.generate_chunks(file_path)
    
    # Save chunks
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    chunks_file = f"/kaggle/working/{base_name}_chunks.json"
    save_chunks_to_file(chunks_data, chunks_file)
    
    # Step 2: Analyze rare regions
    print(f"\n🔍 STEP 2: Analyzing rare regions...")
    rare_regions_file = interactive_analysis(chunks_file)
    
    print(f"\n✅ COMPLETE! Final output: {rare_regions_file}")
    return rare_regions_file

def analyze_from_chunks(chunks_file: str):
    """
    Start from existing chunks file → rare regions only
    
    Args:
        chunks_file: Path to existing filename_chunks.json
    """
    print("🔍 RARE REGION ANALYSIS: Starting from existing chunks")
    print("=" * 55)
    
    if not os.path.exists(chunks_file):
        print(f"❌ Chunks file not found: {chunks_file}")
        return None
    
    print(f"📦 Using existing chunks: {os.path.basename(chunks_file)}")
    rare_regions_file = interactive_analysis(chunks_file)
    
    print(f"\n✅ COMPLETE! Rare regions: {rare_regions_file}")
    return rare_regions_file

def quick_chunks_only(file_path: str, remove_comments: bool = True):
    """
    Generate chunks only (no rare region analysis)
    
    Args:
        file_path: Path to .sv or .v file
        remove_comments: Whether to remove comments (default: True)
    """
    print("📦 CHUNKS ONLY: Generating chunks without rare region analysis")
    print("=" * 60)
    
    generator = SequentialChunkGenerator()
    generator.remove_comments_enabled = remove_comments
    
    chunks_data = generator.generate_chunks(file_path)
    
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    chunks_file = f"/kaggle/working/{base_name}_chunks.json"
    save_chunks_to_file(chunks_data, chunks_file)
    
    print(f"\n✅ Chunks saved: {chunks_file}")
    print("💡 Next step: analyze_from_chunks('{chunks_file}') for rare regions")
    return chunks_file

def check_existing_files(file_path: str):
    """
    Check what files already exist and suggest next steps
    
    Args:
        file_path: Path to .sv or .v file
    """
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    chunks_file = f"/kaggle/working/{base_name}_chunks.json"
    rare_regions_file = f"/kaggle/working/{base_name}_rare_regions.json"
    
    print("🔍 CHECKING EXISTING FILES")
    print("=" * 30)
    print(f"📁 Base file: {os.path.basename(file_path)}")
    
    chunks_exists = os.path.exists(chunks_file)
    rare_regions_exists = os.path.exists(rare_regions_file)
    
    print(f"📦 Chunks file: {os.path.basename(chunks_file)} {'✅ EXISTS' if chunks_exists else '❌ NOT FOUND'}")
    print(f"🔍 Rare regions: {os.path.basename(rare_regions_file)} {'✅ EXISTS' if rare_regions_exists else '❌ NOT FOUND'}")
    
    print(f"\n💡 RECOMMENDED NEXT STEPS:")
    
    if rare_regions_exists:
        print("   🎉 Analysis already complete!")
        print(f"   📋 View results: view_results('{rare_regions_file}')")
    elif chunks_exists:
        print("   📦 Chunks ready - run rare region analysis:")
        print(f"   🔍 analyze_from_chunks('{chunks_file}')")
    else:
        print("   🚀 Start fresh - complete pipeline:")
        print(f"   📝 analyze_file_complete('{file_path}')")
        print("   OR")
        print("   📦 Chunks only first:")
        print(f"   🔧 quick_chunks_only('{file_path}')")
    
    return {
        "chunks_exists": chunks_exists,
        "rare_regions_exists": rare_regions_exists,
        "chunks_file": chunks_file,
        "rare_regions_file": rare_regions_file
    }

# Simple wrapper functions for common use cases
def analyze_file(file_path: str):
    """Simple complete analysis - most common usage"""
    return analyze_file_complete(file_path)

def analyze_chunks(chunks_file: str):
    """Simple rare region analysis from chunks"""
    return analyze_from_chunks(chunks_file)

# Smart auto-detection function
def smart_analyze(file_path: str):
    """
    Smart analysis - automatically detects what to do based on existing files
    
    Args:
        file_path: Path to .sv or .v file
    """
    status = check_existing_files(file_path)
    
    if status["rare_regions_exists"]:
        print(f"\n🎉 Analysis already complete!")
        view_results(status["rare_regions_file"])
        return status["rare_regions_file"]
    elif status["chunks_exists"]:
        print(f"\n📦 Found existing chunks, starting rare region analysis...")
        return analyze_from_chunks(status["chunks_file"])
    else:
        print(f"\n🚀 Starting complete analysis from scratch...")
        return analyze_file_complete(file_path)

# Usage examples and help
print("🎯 FLEXIBLE PIPELINE READY!")
print("=" * 35)
print("")
print("🔧 USAGE OPTIONS:")
print("")
print("1️⃣ COMPLETE ANALYSIS (file → chunks → rare regions):")
print("   analyze_file('/path/to/file.sv')")
print("")
print("2️⃣ RARE REGIONS ONLY (from existing chunks):")
print("   analyze_chunks('/kaggle/working/filename_chunks.json')")
print("")
print("3️⃣ CHUNKS ONLY (no rare region analysis):")
print("   quick_chunks_only('/path/to/file.sv')")
print("")
print("4️⃣ CHECK STATUS (see what files exist):")
print("   check_existing_files('/path/to/file.sv')")
print("")
print("5️⃣ SMART AUTO (detects what to do automatically):")
print("   smart_analyze('/path/to/file.sv')")
print("")
print("📋 VIEW RESULTS:")
print("   view_results('/kaggle/working/filename_rare_regions.json')")
print("")
print("💡 MOST COMMON:")
print("   🚀 smart_analyze('/kaggle/input/riscv-ht/RISCV.sv')")

🎯 FLEXIBLE PIPELINE READY!

🔧 USAGE OPTIONS:

1️⃣ COMPLETE ANALYSIS (file → chunks → rare regions):
   analyze_file('/path/to/file.sv')

2️⃣ RARE REGIONS ONLY (from existing chunks):
   analyze_chunks('/kaggle/working/filename_chunks.json')

3️⃣ CHUNKS ONLY (no rare region analysis):
   quick_chunks_only('/path/to/file.sv')

4️⃣ CHECK STATUS (see what files exist):
   check_existing_files('/path/to/file.sv')

5️⃣ SMART AUTO (detects what to do automatically):
   smart_analyze('/path/to/file.sv')

📋 VIEW RESULTS:
   view_results('/kaggle/working/filename_rare_regions.json')

💡 MOST COMMON:
   🚀 smart_analyze('/kaggle/input/riscv-ht/RISCV.sv')


In [14]:
analyze_chunks('/kaggle/working/RISCV_chunks.json')

🔍 RARE REGION ANALYSIS: Starting from existing chunks
📦 Using existing chunks: RISCV_chunks.json
🚀 Starting analysis of 23 chunks...

🧩 CHUNK: chunk_001
🏗️ TYPE: miscellaneous | 📦 MODULE: None

TASK: Analyze this SystemVerilog code and identify RARE EXECUTION REGIONS.

LOOK FOR any code patterns that execute INFREQUENTLY during normal operation:
• Conditional blocks that trigger only under specific/uncommon conditions
• Error handling paths and exception cases
• Initialization, reset, or startup sequences
• Debug, test, or configuration modes
• Complex loops or computationally expensive operations
• Security checks, privilege validations, or audit trails
• Cache miss handlers, memory fault recovery
• Calibration routines, self-test procedures
• Interrupt handlers for rare events
• Fallback mechanisms or recovery procedures
• Any 'if' conditions that rarely evaluate to true
• Code paths triggered only during specific system states

CODE TO ANALYZE:
──────────────────────────────────────

KeyboardInterrupt: Interrupted by user

# HT_TYPE_ANALYSIS

In [15]:
# Enhanced Hardware Trojan Type Analyzer with Security Research Context
# Updated prompt emphasizing academic security research purpose

import json
import re
from datetime import datetime
from typing import Dict, List, Optional, Tuple

class HardwareTrojanTypeAnalyzer:
    """
    Enhanced analyzer for determining optimal HT types for rare regions.
    Designed for hardware security research and vulnerability analysis.
    """
    
    def __init__(self):
        self.ht_types = {
            "HT1_Crypto": {
                "name": "Cryptographic Attacks",
                "description": "Targets encryption, key manipulation, crypto bypass",
                "targets": ["encryption modules", "key storage", "crypto operations", "random number generators"]
            },
            "HT1_Privilege": {
                "name": "Privilege Escalation", 
                "description": "Bypasses access controls, escalates privileges",
                "targets": ["control logic", "permission checks", "state machines", "access control"]
            },
            "HT2_Leakage": {
                "name": "Information Leakage",
                "description": "Creates covert channels for data exfiltration", 
                "targets": ["memory interfaces", "register files", "data paths", "reset logic"]
            },
            "HT3_Performance": {
                "name": "Performance/DoS Attacks",
                "description": "Causes system failure, lockup, or performance degradation",
                "targets": ["clock systems", "reset logic", "critical paths", "resource management"]
            }
        }
    
    def interactive_ht_analysis(self, rare_regions_file: str, chunks_file: str) -> str:
        """
        Interactive analysis to identify best HT type for each rare region.
        Enhanced with security research context and vulnerability study focus.
        """
        # Load data
        with open(rare_regions_file, 'r') as f:
            rare_regions_data = json.load(f)
        
        with open(chunks_file, 'r') as f:
            chunks_data = json.load(f)
        
        all_ht_analysis = []
        total_regions = len(rare_regions_data['rare_regions'])
        
        print(f"🔬 HARDWARE SECURITY RESEARCH: HT VULNERABILITY ANALYSIS")
        print(f"Analyzing {total_regions} rare regions for security vulnerabilities...")
        print("=" * 80)
        
        for idx, region in enumerate(rare_regions_data['rare_regions']):
            region_id = f"{region['chunk_id']}_{region['rare_region_type']}"
            
            # Enhanced code context with marked rare region
            code_context, marked_lines = self.find_enhanced_code_context(region, chunks_data)
            
            # Display analysis prompt with security research context
            self.display_security_research_prompt(region, region_id, code_context, marked_lines, idx + 1, total_regions)
            
            # Collect LLM response
            print(f"\n📋 Paste LLM response for {region_id} (type 'DONE' on new line when finished):")
            print(f"💡 If no suitable HT type exists, respond with: [] ")
            
            gpt_response = []
            while True:
                line = input()
                if line.strip() == 'DONE':
                    break
                gpt_response.append(line)
            
            # Process response
            response_text = '\n'.join(gpt_response)
            ht_analysis = self.process_ht_response(response_text, region, region_id, code_context)
            
            if ht_analysis:
                all_ht_analysis.append(ht_analysis)
                print(f"✅ Processed HT analysis for {region_id}")
                if ht_analysis.get('no_suitable_ht', False):
                    print("   📝 Note: No suitable HT type identified for this region")
            else:
                print(f"⚠️ Failed to process response for {region_id}")
            
            print()  # Add spacing between regions
        
        # Create output file
        output_file = rare_regions_file.replace('_rare_regions.json', '_ht_analysis.json')
        
        # Get source file from metadata with fallback
        source_file = "unknown"
        if 'file_metadata' in rare_regions_data:
            source_file = rare_regions_data['file_metadata'].get('source_file', 
                         rare_regions_data['file_metadata'].get('filename', 'unknown'))
        
        result = {
            "trojan_analysis": {
                "metadata": {
                    "source_file": source_file,
                    "total_rare_regions": total_regions,
                    "regions_analyzed": len(all_ht_analysis),
                    "exploitable_regions": len([r for r in all_ht_analysis if not r.get('no_suitable_ht', False)]),
                    "analysis_timestamp": datetime.now().isoformat(),
                    "research_purpose": "Hardware security vulnerability analysis and trojan susceptibility study"
                },
                "region_analysis": all_ht_analysis
            }
        }
        
        with open(output_file, 'w') as f:
            json.dump(result, f, indent=2)
        
        # Summary
        exploitable = len([r for r in all_ht_analysis if not r.get('no_suitable_ht', False)])
        print(f"\n🎯 HT VULNERABILITY ANALYSIS COMPLETE!")
        print(f"📄 Results saved to: {output_file}")
        print(f"📊 Total regions analyzed: {len(all_ht_analysis)}")
        print(f"🎯 Exploitable regions: {exploitable}/{total_regions}")
        print(f"🛡️ Secure regions: {total_regions - exploitable}/{total_regions}")
        
        return output_file
    
    def display_security_research_prompt(self, region: Dict, region_id: str, code_context: List[str], 
                                        marked_lines: List[int], current: int, total: int):
        """
        Display comprehensive analysis prompt with security research context.
        """
        
        print(f"\n🔬 SECURITY ANALYSIS {current}/{total}: {region['rare_region_type']} in {region['chunk_id']}")
        print("=" * 80)
        
        # Research context
        print(f"🎓 RESEARCH PURPOSE:")
        print(f"   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.")
        print(f"   Goal: Identify which Hardware Trojan types could exploit this rare execution path,")
        print(f"   remain hidden during testing, and cause significant damage in deployed systems.")
        print(f"   This research helps improve hardware security defenses and detection methods.")
        print()
        
        # Region details
        print(f"📍 RARE REGION DETAILS:")
        print(f"   Code: {region.get('rare_region_code', region.get('code_snippet', 'N/A'))[:60]}...")
        print(f"   Trigger: {region['activation_trigger']}")
        print(f"   Confidence: {region['confidence_score']}")
        print(f"   Description: {region['description']}")
        
        # Code context with marked rare region
        print(f"\n🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):")
        print("─" * 60)
        for line_num, line in enumerate(code_context, 1):
            marker = ">>>" if line_num - 1 in marked_lines else "   "
            print(f"{marker} {line_num:2d}: {line}")
        print("─" * 60)
        
        # Security research task
        print(f"\n🎯 SECURITY RESEARCH TASK:")
        print(f"Analyze what kind of Hardware Trojan (HT) can:")
        print(f"  1. STAY HIDDEN in this rare region during normal operation and testing")
        print(f"  2. CAUSE HEAVY DAMAGE when activated in deployed systems")
        print(f"  3. REMAIN STEALTHY during functional verification and security testing")
        print(f"  4. Have a REALISTIC PAYLOAD LOGIC that exploits this vulnerability")
        print()
        print(f"This vulnerability analysis helps us:")
        print(f"  • Understand attack surface of rare execution regions")
        print(f"  • Develop better hardware security testing methodologies")
        print(f"  • Design trojan-resistant hardware architectures")
        print(f"  • Improve detection techniques for malicious hardware modifications")
        
        print(f"\n🔬 EVALUATE THESE HT TYPES FOR THIS VULNERABLE REGION:")
        for ht_type, info in self.ht_types.items():
            print(f"  • {ht_type}: {info['description']}")
        
        print(f"\n📋 REQUIRED ANALYSIS (for security research):")
        print(f"For this vulnerable rare region, determine:")
        print(f"  1. BEST HT TYPE - Which trojan category fits this vulnerability")
        print(f"  2. DETAILED RATIONALE - WHY this rare region is susceptible to this HT type")
        print(f"  3. PAYLOAD LOGIC - Specific attack mechanism that exploits this rare execution path")
        print(f"  4. STEALTH ASSESSMENT - How the trojan remains undetected during testing")
        print(f"     (Low/Medium/High/Very High)")
        print(f"  5. DAMAGE POTENTIAL - Impact if trojan activates in production")
        print(f"     (Low/Medium/High/Critical)")
        print(f"  6. DETECTION DIFFICULTY - How hard to find using current methods")
        print(f"     (Easy/Medium/Hard/Very Hard/Extremely Hard)")
        
        print(f"\n💡 IMPORTANT FOR RESEARCH:")
        print(f"  • Focus on REALISTIC attack scenarios that could occur in supply chain")
        print(f"  • Consider how rare execution makes trojan harder to detect")
        print(f"  • Explain specific payload logic that exploits this vulnerability")
        print(f"  • If NO suitable HT type exists for this region, respond with: []")
        
        print(f"\n🎯 REQUIRED JSON OUTPUT FORMAT:")
        print('{')
        print('  "best_ht_type": "HT2_Leakage",')
        print('  "rationale": "This rare reset sequence has privileged access to all registers. The infrequent execution makes trojan activation nearly impossible to observe during testing, while still allowing data exfiltration during rare reset events in production.",')
        print('  "attack_method": "Insert additional logic in reset path to sample register contents during reset and transmit them via unused GPIO pins using a timing-based covert channel. Payload activates only during rare reset conditions, making it undetectable during normal testing.",')
        print('  "stealth_level": "Very High",')
        print('  "damage_potential": "Critical",')
        print('  "detection_difficulty": "Extremely Hard"')
        print('}')
        print("\nFor non-exploitable regions: []")
        
        print(f"\n📊 Progress: {current}/{total} regions analyzed")
        print("=" * 80)
    
    def find_enhanced_code_context(self, region: Dict, chunks_data: Dict) -> Tuple[List[str], List[int]]:
        """
        Find enhanced code context with marked rare region lines.
        Returns (context_lines, marked_line_indices)
        """
        chunk_id = region['chunk_id']
        code_snippet = region.get('rare_region_code', region.get('code_snippet', '')).strip()
        
        if chunk_id not in chunks_data['chunks']:
            return [code_snippet], [0]
        
        chunk_content = chunks_data['chunks'][chunk_id]['content']
        lines = chunk_content.split('\n')
        
        # Find target line(s) containing the code snippet
        target_lines = []
        for i, line in enumerate(lines):
            if self.snippet_matches_line(code_snippet, line):
                target_lines.append(i)
        
        if not target_lines:
            target_lines = self.fuzzy_find_snippet(code_snippet, lines)
        
        if not target_lines:
            return [f"// RARE REGION: {code_snippet}"], [0]
        
        # Extract context around target lines
        context_start = max(0, min(target_lines) - 7)
        context_end = min(len(lines), max(target_lines) + 8)
        
        context_lines = []
        marked_lines = []
        
        for i in range(context_start, context_end):
            if i < len(lines):
                line = lines[i].rstrip()
                if i in target_lines:
                    marked_line = f"{line}  // <-- RARE REGION"
                    context_lines.append(marked_line)
                    marked_lines.append(len(context_lines) - 1)
                else:
                    context_lines.append(line)
        
        return context_lines, marked_lines
    
    def snippet_matches_line(self, snippet: str, line: str) -> bool:
        """Check if snippet matches line"""
        clean_snippet = re.sub(r'\s+', ' ', snippet.strip())
        clean_line = re.sub(r'\s+', ' ', line.strip())
        
        if clean_snippet in clean_line:
            return True
        
        snippet_keywords = re.findall(r'\w+', clean_snippet.lower())
        line_keywords = re.findall(r'\w+', clean_line.lower())
        
        if len(snippet_keywords) >= 2:
            matches = sum(1 for kw in snippet_keywords if kw in line_keywords)
            return matches >= len(snippet_keywords) * 0.7
        
        return False
    
    def fuzzy_find_snippet(self, snippet: str, lines: List[str]) -> List[int]:
        """Fuzzy search for snippet in lines"""
        snippet_words = set(re.findall(r'\w+', snippet.lower()))
        if len(snippet_words) < 2:
            return []
        
        best_matches = []
        for i, line in enumerate(lines):
            line_words = set(re.findall(r'\w+', line.lower()))
            common = snippet_words.intersection(line_words)
            
            if len(common) >= len(snippet_words) * 0.6:
                best_matches.append(i)
        
        return best_matches[:3]
    
    def process_ht_response(self, response_text: str, original_region: Dict, 
                          region_id: str, code_context: List[str]) -> Optional[Dict]:
        """Process LLM response into structured analysis"""
        
        # Check for empty/no suitable HT response
        if response_text.strip() in ['[]', '', 'null']:
            return {
                "region_id": region_id,
                "chunk_id": original_region['chunk_id'],
                "original_rare_region": {
                    "rare_region_type": original_region['rare_region_type'],
                    "code_snippet": original_region.get('rare_region_code', original_region.get('code_snippet', 'N/A')),
                    "activation_trigger": original_region['activation_trigger'],
                    "confidence_score": original_region['confidence_score'],
                    "description": original_region['description']
                },
                "code_context": code_context,
                "no_suitable_ht": True,
                "analysis_note": "No suitable hardware trojan type identified for this region"
            }
        
        try:
            # Try to parse JSON response
            if '{' in response_text and '}' in response_text:
                start = response_text.find('{')
                end = response_text.rfind('}') + 1
                json_str = response_text[start:end]
                ht_data = json.loads(json_str)
                
                # Validate required fields
                required_fields = ['best_ht_type', 'rationale', 'attack_method', 
                                 'stealth_level', 'damage_potential', 'detection_difficulty']
                
                if all(field in ht_data for field in required_fields):
                    return {
                        "region_id": region_id,
                        "chunk_id": original_region['chunk_id'],
                        "original_rare_region": {
                            "rare_region_type": original_region['rare_region_type'],
                            "code_snippet": original_region.get('rare_region_code', original_region.get('code_snippet', 'N/A')),
                            "activation_trigger": original_region['activation_trigger'],
                            "confidence_score": original_region['confidence_score'],
                            "description": original_region['description']
                        },
                        "code_context": code_context,
                        "best_ht_match": {
                            "trojan_type": ht_data['best_ht_type'],
                            "rationale": ht_data['rationale'],
                            "attack_method": ht_data['attack_method'],
                            "stealth_level": ht_data['stealth_level'],
                            "damage_potential": ht_data['damage_potential'],
                            "detection_difficulty": ht_data['detection_difficulty']
                        },
                        "no_suitable_ht": False
                    }
        except json.JSONDecodeError:
            pass
        
        print(f"⚠️ Warning: Could not parse response for {region_id}")
        return None

    def view_ht_analysis_results(self, ht_analysis_file: str):
        """View HT analysis results"""
        with open(ht_analysis_file, 'r') as f:
            data = json.load(f)
        
        metadata = data['trojan_analysis']['metadata']
        regions = data['trojan_analysis']['region_analysis']
        
        print("🔬 HARDWARE TROJAN VULNERABILITY ANALYSIS RESULTS")
        print("=" * 70)
        print(f"📁 Source: {metadata['source_file']}")
        print(f"📈 Total Regions: {metadata['total_rare_regions']}")
        print(f"🔍 Analyzed: {metadata['regions_analyzed']}")
        print(f"🎯 Exploitable: {metadata['exploitable_regions']}")
        print(f"🛡️ Secure: {metadata['total_rare_regions'] - metadata['exploitable_regions']}")
        print(f"📅 Analysis Time: {metadata['analysis_timestamp']}")
        
        print(f"\n🎯 VULNERABLE REGIONS (Exploitable by HT):")
        print("-" * 70)
        
        for analysis in regions:
            if not analysis.get('no_suitable_ht', False):
                best_ht = analysis['best_ht_match']
                print(f"\n📦 {analysis['region_id']}:")
                print(f"  🎯 HT Type: {best_ht['trojan_type']}")
                print(f"  💀 Damage: {best_ht['damage_potential']}")
                print(f"  👻 Stealth: {best_ht['stealth_level']}")
                print(f"  🔍 Detection: {best_ht['detection_difficulty']}")
                print(f"  📝 Attack: {best_ht['attack_method'][:80]}...")

    def summarize_ht_distribution(self, ht_analysis_file: str):
        """Summarize HT type distribution"""
        with open(ht_analysis_file, 'r') as f:
            data = json.load(f)
        
        regions = data['trojan_analysis']['region_analysis']
        exploitable = [r for r in regions if not r.get('no_suitable_ht', False)]
        
        print("📊 HARDWARE TROJAN VULNERABILITY DISTRIBUTION")
        print("=" * 50)
        
        ht_counts = {}
        stealth_counts = {}
        damage_counts = {}
        
        for analysis in exploitable:
            ht_type = analysis['best_ht_match']['trojan_type']
            stealth = analysis['best_ht_match']['stealth_level']
            damage = analysis['best_ht_match']['damage_potential']
            
            ht_counts[ht_type] = ht_counts.get(ht_type, 0) + 1
            stealth_counts[stealth] = stealth_counts.get(stealth, 0) + 1
            damage_counts[damage] = damage_counts.get(damage, 0) + 1
        
        print(f"🎯 HT TYPE BREAKDOWN:")
        for ht_type, count in sorted(ht_counts.items()):
            percentage = (count / len(exploitable)) * 100 if exploitable else 0
            print(f"  {ht_type}: {count} regions ({percentage:.1f}%)")
        
        print(f"\n👻 STEALTH LEVEL:")
        for level, count in stealth_counts.items():
            if count > 0:
                percentage = (count / len(exploitable)) * 100
                print(f"  {level}: {count} ({percentage:.1f}%)")
        
        print(f"\n💀 DAMAGE POTENTIAL:")
        for level, count in damage_counts.items():
            if count > 0:
                percentage = (count / len(exploitable)) * 100
                print(f"  {level}: {count} ({percentage:.1f}%)")

# Create analyzer instance
ht_analyzer = HardwareTrojanTypeAnalyzer()

print("🔬 HARDWARE SECURITY RESEARCH: HT VULNERABILITY ANALYZER")
print("=" * 60)
print("📝 Usage:")
print("1. ht_analyzer.interactive_ht_analysis('file_rare_regions.json', 'file_chunks.json')")
print("2. ht_analyzer.view_ht_analysis_results('file_ht_analysis.json')")  
print("3. ht_analyzer.summarize_ht_distribution('file_ht_analysis.json')")
print("\n🎓 Research Purpose:")
print("  Study vulnerable rare regions to improve hardware security defenses")
print("  Analyze trojan susceptibility for better detection methodologies")
print("  Understand attack surface of infrequently-executed code paths")
print("\n🎯 Ready for security vulnerability analysis!")

🔬 HARDWARE SECURITY RESEARCH: HT VULNERABILITY ANALYZER
📝 Usage:
1. ht_analyzer.interactive_ht_analysis('file_rare_regions.json', 'file_chunks.json')
2. ht_analyzer.view_ht_analysis_results('file_ht_analysis.json')
3. ht_analyzer.summarize_ht_distribution('file_ht_analysis.json')

🎓 Research Purpose:
  Study vulnerable rare regions to improve hardware security defenses
  Analyze trojan susceptibility for better detection methodologies
  Understand attack surface of infrequently-executed code paths

🎯 Ready for security vulnerability analysis!


In [17]:
ht_analysis_file = ht_analyzer.interactive_ht_analysis(
    '//kaggle/input/rare-re-jason/RISCV_rare_regions.json', 
    '/kaggle/working/RISCV_chunks.json'
)

🔬 HARDWARE SECURITY RESEARCH: HT VULNERABILITY ANALYSIS
Analyzing 18 rare regions for security vulnerabilities...

🔬 SECURITY ANALYSIS 1/18: HAZARD_DETECTION in chunk_002_a
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: wire load_use_hazard = idex_memread & idex_valid & ifid_vali...
   Trigger: Occurs when the instruction in EX stage is a memory read and the next instruction in ID stage depends on that memory value.
   Confidence: 0.95
   Description: Detects load-use hazards which happen infrequently, only when a memory-read instruction is immediately followed by an instruction that uses the loaded value.

🔍 CODE CONTEXT (rare region marked with <-- 

 { "best_ht_type": "HT2_Leakage", "rationale": "The load-use hazard is a narrow, infrequently exercised window where a freshly loaded value is live in the pipeline and the following instruction consumes it. That brief, data-rich interval gives a trojan a chance to observe or subtly influence a sensitive datum (for example a key, pointer, or control word) without touching common, heavily-tested datapaths. Because the condition (memory-read immediately followed by dependent instruction) is rare in typical benchmarks and standard functional tests, a leakage trojan triggered there can remain dormant during verification and only exfiltrate when the rare pattern occurs in the field.", "attack_method": "High-level: add a tiny, dormant datapath or timing modifier that is only enabled when the load-use hazard signal is asserted. When enabled, the logic creates a small, data-dependent observable effect (for example a subtle timing variation, a microarchitectural side effect, or controlled conten

✅ Processed HT analysis for chunk_002_a_HAZARD_DETECTION


🔬 SECURITY ANALYSIS 2/18: FORWARDING_LOGIC in chunk_002_a
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: assign ex_op_a = (fwd_a_sel==2'd0) ? idex_r1 :
             ...
   Trigger: Triggered only when forwarding is required due to data dependencies, which is infrequent under typical instruction sequences.
   Confidence: 0.9
   Description: Handles rare data forwarding cases to resolve hazards when EX stage operands depend on results from later pipeline stages.

🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):
────────────────────────────────────────────────────────────
     1:        

 { "best_ht_type": "HT1_Privilege", "rationale": "The forwarding-path case for memwb (fwd_sel==2) is exercised only when an instruction in EX must take a value written back in the same cycle — a relatively rare timing window. That path can carry ALU results, loaded memory data, or pc_plus4 (return addresses). A tiny malicious modification that selectively alters the value forwarded from memwb (only when memwb_wb_sel==2 or when the destination register is a link/return register) can silently corrupt return addresses or critical control data. Corrupting those values enables control-flow hijacks or privilege-escalation sequences while touching code paths that are seldom triggered in tests, making this region highly attractive for privilege-focused Trojans.", "attack_method": "Payload implants a compact conditional datapath in the memwb->ex forwarding mux: when (fwd_a_sel==2 || fwd_b_sel==2) AND (memwb_wb_sel==2 OR memwb_rd==<target_link_reg>) AND a rare trigger pattern (e.g., a specific i

✅ Processed HT analysis for chunk_002_a_FORWARDING_LOGIC


🔬 SECURITY ANALYSIS 3/18: BRANCH_TAKING in chunk_002_a
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: assign pc_next = take_branch_ex ? branch_target_ex : (pc_if ...
   Trigger: Occurs only when a branch is taken, which is relatively infrequent compared to sequential instruction execution.
   Confidence: 0.85
   Description: Controls program counter update for branch instructions; most instructions do not alter PC via branches.

🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):
────────────────────────────────────────────────────────────
     1: );
     2: 
     3:     reg  [31:0] pc_i

 { "best_ht_type": "HT1_Privilege", "rationale": "The pc update path is a direct control-flow choke point: changing pc_next when a branch is taken immediately redirects program execution. Branch-taking is relatively infrequent and exercised differently in tests vs. real workloads, so a tiny modification here can remain dormant in verification yet, when triggered, hijack control flow to privileged or malicious code (e.g., a hidden firmware routine or a jump into a ROP-style gadget region). That makes this rare branch-taking window an ideal insertion point for privilege-escalation or control-flow hijack Trojans.", "attack_method": "Implant a compact conditional override in the pc_next logic: when take_branch_ex is true AND a rare trigger condition holds (example triggers: a specific sequence of branch targets, a secret-dependent bit pattern in branch_target_ex or in a small LFSR/sequence counter, or after N occurrences tracked by a tiny counter), replace branch_target_ex with an attacker

... (28 outputs left out)

✅ Processed HT analysis for chunk_020_MEMORY_INITIALIZATION


🔬 SECURITY ANALYSIS 18/18: PARTIAL_WRITE_HANDLING in chunk_020
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: if (dmem_req && dmem_we) begin
                if (dmem_be[0...
   Trigger: Triggered when a data memory write occurs (dmem_req && dmem_we) with specific byte enables; partial writes are less frequent than full word writes.
   Confidence: 0.9
   Description: Handles selective byte-wise writing to memory, rarely activated when only some bytes need updating.

🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):
────────────────────────────────────────────────────────────
>>>  1: 

 { "best_ht_type": "HT1_Privilege", "rationale": "The memory-initialization code runs only during reset and has authoritative control over large memory contents and (indirectly) boot behavior. A malicious change here can deterministically shape the system's initial software view (boot vectors, firmware image placement, configuration words) before any higher-level software or checks run. Because resets are infrequent and many verification flows do not exercise exhaustive memory-init variants, implants that alter initialization can remain dormant through standard testing yet guarantee attacker control the first time the device boots in the field.", "attack_method": "High-level (non-actionable): modify the reset-time initialization so that one or more memory locations are left with attacker-chosen values (instead of zero) or are seeded with a tiny bootstrap sequence. On first boot the processor fetches or executes from these modified locations (or reads attacker-chosen configuration words

✅ Processed HT analysis for chunk_020_PARTIAL_WRITE_HANDLING



OSError: [Errno 30] Read-only file system: '//kaggle/input/rare-re-jason/RISCV_ht_analysis.json'

In [20]:
import shutil

# Copy from input to working
shutil.copy(
    '/kaggle/input/rare-re-jason/RISCV_rare_regions.json',
    '/kaggle/working/RISCV_rare_regions.json'
)

# Now run analysis using working directory
ht_analysis_file = ht_analyzer.interactive_ht_analysis(
    '/kaggle/working/RISCV_rare_regions.json',
    '/kaggle/working/RISCV_chunks.json'
)

🔬 HARDWARE SECURITY RESEARCH: HT VULNERABILITY ANALYSIS
Analyzing 18 rare regions for security vulnerabilities...

🔬 SECURITY ANALYSIS 1/18: HAZARD_DETECTION in chunk_002_a
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: wire load_use_hazard = idex_memread & idex_valid & ifid_vali...
   Trigger: Occurs when the instruction in EX stage is a memory read and the next instruction in ID stage depends on that memory value.
   Confidence: 0.95
   Description: Detects load-use hazards which happen infrequently, only when a memory-read instruction is immediately followed by an instruction that uses the loaded value.

🔍 CODE CONTEXT (rare region marked with <-- 

 { "best_ht_type": "HT2_Leakage", "rationale": "The load-use hazard is a narrow, infrequently exercised window where a freshly loaded value is live in the pipeline and the following instruction consumes it. That brief, data-rich interval gives a trojan a chance to observe or subtly influence a sensitive datum (for example a key, pointer, or control word) without touching common, heavily-tested datapaths. Because the condition (memory-read immediately followed by dependent instruction) is rare in typical benchmarks and standard functional tests, a leakage trojan triggered there can remain dormant during verification and only exfiltrate when the rare pattern occurs in the field.", "attack_method": "High-level: add a tiny, dormant datapath or timing modifier that is only enabled when the load-use hazard signal is asserted. When enabled, the logic creates a small, data-dependent observable effect (for example a subtle timing variation, a microarchitectural side effect, or controlled conten

✅ Processed HT analysis for chunk_002_a_HAZARD_DETECTION


🔬 SECURITY ANALYSIS 2/18: FORWARDING_LOGIC in chunk_002_a
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: assign ex_op_a = (fwd_a_sel==2'd0) ? idex_r1 :
             ...
   Trigger: Triggered only when forwarding is required due to data dependencies, which is infrequent under typical instruction sequences.
   Confidence: 0.9
   Description: Handles rare data forwarding cases to resolve hazards when EX stage operands depend on results from later pipeline stages.

🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):
────────────────────────────────────────────────────────────
     1:        

 { "best_ht_type": "HT1_Privilege", "rationale": "The forwarding-path case for memwb (fwd_sel==2) is exercised only when an instruction in EX must take a value written back in the same cycle — a relatively rare timing window. That path can carry ALU results, loaded memory data, or pc_plus4 (return addresses). A tiny malicious modification that selectively alters the value forwarded from memwb (only when memwb_wb_sel==2 or when the destination register is a link/return register) can silently corrupt return addresses or critical control data. Corrupting those values enables control-flow hijacks or privilege-escalation sequences while touching code paths that are seldom triggered in tests, making this region highly attractive for privilege-focused Trojans.", "attack_method": "Payload implants a compact conditional datapath in the memwb->ex forwarding mux: when (fwd_a_sel==2 || fwd_b_sel==2) AND (memwb_wb_sel==2 OR memwb_rd==<target_link_reg>) AND a rare trigger pattern (e.g., a specific i

✅ Processed HT analysis for chunk_002_a_FORWARDING_LOGIC


🔬 SECURITY ANALYSIS 3/18: BRANCH_TAKING in chunk_002_a
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: assign pc_next = take_branch_ex ? branch_target_ex : (pc_if ...
   Trigger: Occurs only when a branch is taken, which is relatively infrequent compared to sequential instruction execution.
   Confidence: 0.85
   Description: Controls program counter update for branch instructions; most instructions do not alter PC via branches.

🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):
────────────────────────────────────────────────────────────
     1: );
     2: 
     3:     reg  [31:0] pc_i

 { "best_ht_type": "HT1_Privilege", "rationale": "The pc update path is a direct control-flow choke point: changing pc_next when a branch is taken immediately redirects program execution. Branch-taking is relatively infrequent and exercised differently in tests vs. real workloads, so a tiny modification here can remain dormant in verification yet, when triggered, hijack control flow to privileged or malicious code (e.g., a hidden firmware routine or a jump into a ROP-style gadget region). That makes this rare branch-taking window an ideal insertion point for privilege-escalation or control-flow hijack Trojans.", "attack_method": "Implant a compact conditional override in the pc_next logic: when take_branch_ex is true AND a rare trigger condition holds (example triggers: a specific sequence of branch targets, a secret-dependent bit pattern in branch_target_ex or in a small LFSR/sequence counter, or after N occurrences tracked by a tiny counter), replace branch_target_ex with an attacker

... (27 outputs left out)

 { "best_ht_type": "HT1_Privilege", "rationale": "The memory-initialization code runs only during reset and has authoritative control over large memory contents and (indirectly) boot behavior. A malicious change here can deterministically shape the system's initial software view (boot vectors, firmware image placement, configuration words) before any higher-level software or checks run. Because resets are infrequent and many verification flows do not exercise exhaustive memory-init variants, implants that alter initialization can remain dormant through standard testing yet guarantee attacker control the first time the device boots in the field.", "attack_method": "High-level (non-actionable): modify the reset-time initialization so that one or more memory locations are left with attacker-chosen values (instead of zero) or are seeded with a tiny bootstrap sequence. On first boot the processor fetches or executes from these modified locations (or reads attacker-chosen configuration words

✅ Processed HT analysis for chunk_020_MEMORY_INITIALIZATION


🔬 SECURITY ANALYSIS 18/18: PARTIAL_WRITE_HANDLING in chunk_020
🎓 RESEARCH PURPOSE:
   This analysis is for HARDWARE SECURITY RESEARCH to study vulnerable rare regions.
   Goal: Identify which Hardware Trojan types could exploit this rare execution path,
   remain hidden during testing, and cause significant damage in deployed systems.
   This research helps improve hardware security defenses and detection methods.

📍 RARE REGION DETAILS:
   Code: if (dmem_req && dmem_we) begin
                if (dmem_be[0...
   Trigger: Triggered when a data memory write occurs (dmem_req && dmem_we) with specific byte enables; partial writes are less frequent than full word writes.
   Confidence: 0.9
   Description: Handles selective byte-wise writing to memory, rarely activated when only some bytes need updating.

🔍 CODE CONTEXT (rare region marked with <-- RARE REGION):
────────────────────────────────────────────────────────────
>>>  1: 

 { "best_ht_type": "HT1_Privilege", "rationale": "Partial-byte write handling is a narrow, infrequently exercised datapath that decides which bytes within a word are updated. A malicious, narrowly-scoped change here can produce one- or two-byte corruptions at precise memory offsets (for example low-order pointer or return-address bytes, ACL flags, or boot configuration fields). Because partial writes occur far less often than full-word writes in many workloads and verification suites, a trojan confined to this path can remain dormant during standard testing yet produce deterministic, security-critical memory corruptions in the field that lead to privilege escalation or persistent compromise.", "attack_method": "Conceptual (non-actionable): silently alter the value written for a targeted byte-enable pattern under very rare, tightly-gated conditions so that a single byte (or a small subset of bytes) in a security-sensitive location is corrupted while other memory traffic remains correct.

✅ Processed HT analysis for chunk_020_PARTIAL_WRITE_HANDLING


🎯 HT VULNERABILITY ANALYSIS COMPLETE!
📄 Results saved to: /kaggle/working/RISCV_ht_analysis.json
📊 Total regions analyzed: 16
🎯 Exploitable regions: 16/18
🛡️ Secure regions: 2/18
